# Colab Session: nemotron-t4-1030
Generated from colab-cli history log.

**Session Created**: 2026-10-30 05:29:39
- Endpoint: `gpu-t4-hm-euw4a0-zohn8xygede1`
- Hardware: `T4`

In [ ]:
import json
import pathlib
import platform
import shutil
import subprocess
import sys

import torch

out = pathlib.Path('/content/nemotron-transcribe')
out.mkdir(exist_ok=True)
facts = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_version': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout,
    'memory': pathlib.Path('/proc/meminfo').read_text(),
    'disk': dict(zip(['total', 'used', 'free'], shutil.disk_usage('/content'))),
}
(out / 'runtime-initial.json').write_text(json.dumps(facts, ensure_ascii=False, indent=2))
print(json.dumps(facts, ensure_ascii=False, indent=2), flush=True)


{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu130",
  "cuda_version": "13.0",
  "cuda_available": true,
  "gpu": "Fri Oct 30 05:30:15 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |                        |               MIG M. |\n|=========================================+========================+======================|\n|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |\n| N/A   44C 

In [ ]:
import importlib.util
import json
import pathlib
import subprocess
import sys

if importlib.util.find_spec('huggingface_hub') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'], check=True)
root = pathlib.Path('/content/nemotron-transcribe')
root.mkdir(exist_ok=True)
source = '''import json, pathlib, time, traceback
from huggingface_hub import snapshot_download
out = pathlib.Path('/content/nemotron-transcribe')
state = {'model_id': 'nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B', 'revision': 'c17667641642c4265ff2b7037dd101b7d4567634', 'started_at': time.time(), 'status': 'downloading'}
(out / 'download-status.json').write_text(json.dumps(state))
try:
    state['snapshot'] = snapshot_download(state['model_id'], revision=state['revision'], cache_dir='/content/hf-cache', max_workers=8, token=False)
    state['status'] = 'complete'
except Exception:
    state['status'] = 'failed'
    state['error'] = traceback.format_exc()
    raise
finally:
    state['finished_at'] = time.time()
    (out / 'download-status.json').write_text(json.dumps(state, indent=2))
    print(json.dumps(state), flush=True)
'''
(root / 'download_model.py').write_text(source)
log = (root / 'download.log').open('w')
process = subprocess.Popen([sys.executable, '-u', str(root / 'download_model.py')], stdout=log, stderr=subprocess.STDOUT)
print(json.dumps({'download_pid': process.pid, 'log': str(root / 'download.log')}), flush=True)



{"download_pid": 2343, "log": "/content/nemotron-transcribe/download.log"}


In [ ]:
#!/usr/bin/env python3
"""Capture the Colab runtime and install the minimal HF inference stack."""

from __future__ import annotations

import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


MODEL_ID = "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B"
MODEL_REVISION = "c17667641642c4265ff2b7037dd101b7d4567634"
TRANSFORMERS_REPOSITORY = "https://github.com/huggingface/transformers.git"
TRANSFORMERS_REVISION = "e82b00dd04d71b2e2801ecf3029eea3858868091"
OUTPUT_DIR = Path(os.environ.get("NEMOTRON_OUT_DIR", "/content/nemotron-transcribe"))
HF_CACHE_DIR = Path(os.environ.get("HF_HOME", "/content/hf-cache"))
RUNTIME_JSON = OUTPUT_DIR / "runtime.json"

PACKAGE_NAMES = (
    "torch",
    "transformers",
    "accelerate",
    "datasets",
    "soundfile",
    "huggingface-hub",
    "safetensors",
    "tokenizers",
    "pyarrow",
)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def package_versions() -> dict[str, str | None]:
    versions: dict[str, str | None] = {}
    for name in PACKAGE_NAMES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def read_memory_facts() -> dict[str, int | None]:
    facts: dict[str, int | None] = {"total_bytes": None, "available_bytes": None}
    meminfo = Path("/proc/meminfo")
    if not meminfo.is_file():
        return facts
    values: dict[str, int] = {}
    for line in meminfo.read_text(encoding="utf-8").splitlines():
        key, _, remainder = line.partition(":")
        if key in {"MemTotal", "MemAvailable"}:
            fields = remainder.split()
            if fields:
                values[key] = int(fields[0]) * 1024
    facts["total_bytes"] = values.get("MemTotal")
    facts["available_bytes"] = values.get("MemAvailable")
    return facts


def collect_environment() -> dict[str, Any]:
    import torch

    gpu_devices: list[dict[str, Any]] = []
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            free_bytes, total_bytes = torch.cuda.mem_get_info(index)
            gpu_devices.append(
                {
                    "index": index,
                    "name": props.name,
                    "capability": list(torch.cuda.get_device_capability(index)),
                    "total_memory_bytes": int(total_bytes),
                    "free_memory_bytes": int(free_bytes),
                }
            )

    disk_path = Path("/content") if Path("/content").exists() else OUTPUT_DIR.parent
    disk = shutil.disk_usage(disk_path)
    nvidia_smi: dict[str, Any] = {"available": shutil.which("nvidia-smi") is not None}
    if nvidia_smi["available"]:
        try:
            completed = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=name,memory.total,memory.free,driver_version",
                    "--format=csv,noheader",
                ],
                check=False,
                capture_output=True,
                text=True,
                timeout=10,
            )
            nvidia_smi.update(
                {
                    "returncode": completed.returncode,
                    "stdout": completed.stdout[-4000:],
                    "stderr": completed.stderr[-2000:],
                }
            )
        except Exception as exc:  # Runtime fact collection is best-effort at this boundary.
            nvidia_smi["error"] = f"{type(exc).__name__}: {exc}"

    return {
        "captured_at_utc": utc_now(),
        "python": {
            "version": platform.python_version(),
            "implementation": platform.python_implementation(),
            "executable": sys.executable,
        },
        "platform": platform.platform(),
        "torch": {
            "version": torch.__version__,
            "cuda_version": torch.version.cuda,
            "cuda_available": torch.cuda.is_available(),
            "device_count": torch.cuda.device_count(),
            "devices": gpu_devices,
        },
        "ram": read_memory_facts(),
        "disk": {
            "path": str(disk_path),
            "total_bytes": disk.total,
            "used_bytes": disk.used,
            "free_bytes": disk.free,
        },
        "nvidia_smi": nvidia_smi,
    }


def write_report(report: dict[str, Any]) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=OUTPUT_DIR,
            prefix="runtime.",
            suffix=".tmp",
            delete=False,
        ) as stream:
            temporary_path = Path(stream.name)
            json.dump(report, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        temporary_path.replace(RUNTIME_JSON)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()


def log(message: str) -> None:
    print(f"[bootstrap] {message}", flush=True)


def run_install(torch_version: str) -> dict[str, Any]:
    constraint_file = OUTPUT_DIR / "torch-constraint.txt"
    constraint_file.write_text(f"torch=={torch_version}\n", encoding="utf-8")
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        "--constraint",
        str(constraint_file),
        f"git+{TRANSFORMERS_REPOSITORY}@{TRANSFORMERS_REVISION}",
        "accelerate>=1.0",
        "datasets>=4.4",
        "soundfile>=0.13",
        "bitsandbytes",
    ]
    log("Installing the pinned Transformers source and audio/data dependencies; torch is constrained to its existing version.")
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    install_result = {
        "command": command,
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout[-12000:],
        "stderr_tail": completed.stderr[-12000:],
    }
    if completed.returncode != 0:
        raise RuntimeError(f"pip install exited with code {completed.returncode}")
    return install_result


def main() -> int:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(HF_CACHE_DIR)
    os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

    report: dict[str, Any] = {
        "schema_version": 1,
        "status": "running",
        "started_at_utc": utc_now(),
        "finished_at_utc": None,
        "model": {"repo_id": MODEL_ID, "revision": MODEL_REVISION},
        "transformers_source": {
            "repository": TRANSFORMERS_REPOSITORY,
            "revision": TRANSFORMERS_REVISION,
        },
        "paths": {
            "output_dir": str(OUTPUT_DIR),
            "hf_cache_dir": str(HF_CACHE_DIR),
            "weights_are_inside_output_dir": False,
        },
        "packages_before": package_versions(),
        "environment_before": None,
        "install": None,
        "packages_after": None,
        "environment_after": None,
        "stages": [],
        "error": None,
    }
    current_stage: dict[str, Any] | None = None

    try:
        log("Collecting the current runtime facts.")
        environment_before = collect_environment()
        report["environment_before"] = environment_before
        report["stages"].append(
            {
                "name": "collect_runtime_before",
                "status": "complete",
                "finished_at_utc": utc_now(),
            }
        )
        write_report(report)

        if not environment_before["torch"]["cuda_available"]:
            raise RuntimeError("CUDA is unavailable; refusing to prepare a CPU-only experiment.")

        current_stage = {
            "name": "install_dependencies",
            "status": "running",
            "started_at_utc": utc_now(),
        }
        report["stages"].append(current_stage)
        write_report(report)
        report["install"] = run_install(environment_before["torch"]["version"])
        importlib.invalidate_caches()
        report["packages_after"] = package_versions()
        environment_after = collect_environment()
        report["environment_after"] = environment_after

        if environment_after["torch"]["version"] != environment_before["torch"]["version"]:
            raise RuntimeError(
                "The dependency installer changed torch from "
                f"{environment_before['torch']['version']} to {environment_after['torch']['version']}."
            )
        if not environment_after["torch"]["cuda_available"]:
            raise RuntimeError("CUDA became unavailable after dependency installation.")

        current_stage["status"] = "complete"
        current_stage["finished_at_utc"] = utc_now()
        report["status"] = "complete"
        report["finished_at_utc"] = utc_now()
        write_report(report)
        log(f"Bootstrap complete. Runtime report: {RUNTIME_JSON}")
        return 0
    except Exception as exc:
        if current_stage is not None and current_stage.get("status") == "running":
            current_stage["status"] = "failed"
            current_stage["finished_at_utc"] = utc_now()
        report["status"] = "failed"
        report["finished_at_utc"] = utc_now()
        report["error"] = {
            "stage": current_stage.get("name") if current_stage else "collect_runtime_before",
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc()[-12000:],
        }
        try:
            report["packages_after"] = package_versions()
            if report["environment_after"] is None:
                report["environment_after"] = collect_environment()
        except Exception as capture_exc:
            report["error"]["post_failure_capture_error"] = f"{type(capture_exc).__name__}: {capture_exc}"
        try:
            write_report(report)
        except Exception as write_exc:
            print(f"[bootstrap] Could not write {RUNTIME_JSON}: {write_exc}", file=sys.stderr, flush=True)
        log(f"Bootstrap failed during {report['error']['stage']}: {type(exc).__name__}: {exc}")
        raise


if __name__ == "__main__":
    main()



[bootstrap] Collecting the current runtime facts.


[bootstrap] Installing the pinned Transformers source and audio/data dependencies; torch is constrained to its existing version.


[bootstrap] Bootstrap complete. Runtime report: /content/nemotron-transcribe/runtime.json


In [ ]:
import json
import pathlib

root = pathlib.Path('/content/nemotron-transcribe')
for name in ['download-status.json', 'runtime.json', 'model-metadata.json', 'results.json']:
    file = root / name
    if file.exists():
        print(name, file.read_text()[-3500:], flush=True)
cache = pathlib.Path('/content/hf-cache')
print('cache_bytes', sum(p.stat().st_size for p in cache.rglob('*') if p.is_file()), flush=True)
for name in ['download.log', 'bootstrap.log', 'transcribe.log']:
    file = root / name
    if file.exists():
        print(name, file.read_text(errors='replace')[-1800:], flush=True)


download-status.json {
  "model_id": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B",
  "revision": "c17667641642c4265ff2b7037dd101b7d4567634",
  "started_at": 1793338233.9273381,
  "status": "complete",
  "snapshot": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634",
  "finished_at": 1793338418.9939663
}


runtime.json 0.0.5)\nRequirement already satisfied: markdown-it-py>=2.2.0 in /usr/local/lib/python3.13/dist-packages (from rich>=13.8.0->typer->transformers==5.22.0.dev0) (4.2.0)\nRequirement already satisfied: pygments<3.0.0,>=2.13.0 in /usr/local/lib/python3.13/dist-packages (from rich>=13.8.0->typer->transformers==5.22.0.dev0) (2.21.0)\nRequirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.13/dist-packages (from markdown-it-py>=2.2.0->rich>=13.8.0->typer->transformers==5.22.0.dev0) (0.1.2)\nDownloading bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)\n   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 7.9 MB/s  0:00:05\nBuilding wheels for collected packages: transformers\n  Building wheel for transformers (pyproject.toml): started\n  Building wheel for transformers (pyproject.toml): finished with status 'done'\n  Created wheel for transformers: filename=transformers-5.22.0.dev0-py3-none-any.whl size=14376170 sha256=cc1c8ae4c7df262638f34182e11a49a

cache_bytes 202548051595


download.log 
Fetching 22 files:   5%|▍         | 1/22 [00:00<00:03,  6.87it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 22 files: 100%|██████████| 22/22 [03:04<00:00,  8.40s/it]
{"model_id": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B", "revision": "c17667641642c4265ff2b7037dd101b7d4567634", "started_at": 1793338233.9273381, "status": "complete", "snapshot": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "finished_at": 1793338418.9939663}



In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/nemo-speech')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/NeMo-Speech.cpp.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', 'd8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14'], check=True)
subprocess.run(['git', '-C', str(repo), 'submodule', 'update', '--init', '--depth', '1', 'llama.cpp'], check=True)
print('Pinned source checkout and llama.cpp submodule are ready.', flush=True)


Pinned source checkout and llama.cpp submodule are ready.


*File Operation*: `upload` on `/content/fleurs-ja-01.wav`

*File Operation*: `upload` on `/content/fleurs-ja-02.wav`

In [ ]:
import subprocess

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake', 'ninja-build', 'git', 'pkg-config', 'libsentencepiece-dev'], check=True)
print('C++ build dependencies installed inside Colab.', flush=True)


C++ build dependencies installed inside Colab.


In [ ]:
import json, pathlib, subprocess, sys
source = json.loads(r'''"#!/usr/bin/env python3\n\"\"\"Prepare pinned NeMo-Speech.cpp, build T4 CUDA SALM, and convert a local checkpoint.\"\"\"\n\nfrom __future__ import annotations\n\nimport importlib.metadata\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n\nREVISION = \"d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14\"\nREPO = Path(\"/content/nemo-speech\")\nSOURCE = Path(\"/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634\")\nOUT = Path(\"/content/nemotron-gguf\")\nREPORT_DIR = Path(\"/content/nemotron-transcribe\")\nSTATUS = REPORT_DIR / \"cpp-status.json\"\nLOG = REPORT_DIR / \"prepare_cpp.log\"\nPATCH_FILE = REPORT_DIR / \"n_gpu_layers.patch\"\nDECODER = OUT / \"decoder-IQ4_NL.gguf\"\n\n\ndef utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef save_status() -> None:\n    REPORT_DIR.mkdir(parents=True, exist_ok=True)\n    STATUS.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(\"w\", encoding=\"utf-8\", dir=REPORT_DIR, delete=False) as f:\n        json.dump(state, f, ensure_ascii=False, indent=2)\n        f.write(\"\\n\")\n        f.flush()\n        os.fsync(f.fileno())\n        tmp = Path(f.name)\n    tmp.replace(STATUS)\n\n\ndef run_stage(name: str, command: list[str], cwd: Path | None = None) -> None:\n    step = {\"status\": \"running\", \"started_at_utc\": utc(), \"command\": command}\n    state[\"phase\"] = name\n    state[\"stages\"][name] = step\n    save_status()\n    print(f\"[prepare_cpp] start {name}: {' '.join(command)}\", flush=True)\n    try:\n        with LOG.open(\"a\", encoding=\"utf-8\", buffering=1) as log:\n            log.write(f\"\\n[{utc()}] {name}: {' '.join(command)}\\n\")\n            process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            assert process.stdout is not None\n            for line in process.stdout:\n                print(line, end=\"\", flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        if code:\n            raise subprocess.CalledProcessError(code, command)\n        step[\"status\"] = \"complete\"\n        step[\"finished_at_utc\"] = utc()\n        save_status()\n    except Exception as exc:\n        step.update({\"status\": \"failed\", \"finished_at_utc\": utc(), \"error\": f\"{type(exc).__name__}: {exc}\"})\n        save_status()\n        raise\n\n\ndef patch_gpu_layers() -> None:\n    name = \"patch_gpu_layers\"\n    state[\"phase\"] = name\n    state[\"stages\"][name] = {\"status\": \"running\", \"started_at_utc\": utc()}\n    save_status()\n    path = REPO / \"src/speechlm/salm_engine.cpp\"\n    source = path.read_text(encoding=\"utf-8\")\n    old = \"        mp.n_gpu_layers = cfg.gpu >= 0 ? 999 : 0;\"\n    new = '''        const char* gpu_layers_env = std::getenv(\"NEMOTRON_GPU_LAYERS\");\n        if (gpu_layers_env) {\n            char* parsed_end = nullptr;\n            const long gpu_layers = std::strtol(gpu_layers_env, &parsed_end, 10);\n            if (parsed_end == gpu_layers_env || *parsed_end != '\\\\0' || gpu_layers < 0 || gpu_layers > 999)\n                throw std::invalid_argument(\"NEMOTRON_GPU_LAYERS must be an integer from 0 to 999\");\n            mp.n_gpu_layers = static_cast<int>(gpu_layers);\n        } else {\n            mp.n_gpu_layers = cfg.gpu >= 0 ? 999 : 0;\n        }'''\n    if \"NEMOTRON_GPU_LAYERS\" not in source:\n        if source.count(old) != 1:\n            raise RuntimeError(\"Pinned SALM source no longer has exactly one expected n_gpu_layers assignment.\")\n        tmp = path.with_suffix(\".cpp.tmp\")\n        tmp.write_text(source.replace(old, new, 1), encoding=\"utf-8\")\n        tmp.replace(path)\n    elif \"mp.n_gpu_layers = static_cast<int>(gpu_layers);\" not in source:\n        raise RuntimeError(\"Existing NEMOTRON_GPU_LAYERS patch does not match the expected implementation.\")\n    diff = subprocess.run([\"git\", \"diff\", \"--binary\", \"--\", \"src/speechlm/salm_engine.cpp\"],\n                          cwd=REPO, check=True, capture_output=True, text=True).stdout\n    if not diff:\n        raise RuntimeError(\"No source diff was produced for the GPU-layer override.\")\n    PATCH_FILE.write_text(diff, encoding=\"utf-8\")\n    state[\"patch_file\"] = str(PATCH_FILE)\n    state[\"stages\"][name].update({\"status\": \"complete\", \"finished_at_utc\": utc(), \"diff_bytes\": len(diff.encode())})\n    save_status()\n    print(f\"[prepare_cpp] saved source diff: {PATCH_FILE}\", flush=True)\n\n\ndef main() -> int:\n    REPORT_DIR.mkdir(parents=True, exist_ok=True)\n    OUT.mkdir(parents=True, exist_ok=True)\n    global state\n    state = {\"status\": \"running\", \"phase\": \"startup\", \"started_at_utc\": utc(), \"revision\": REVISION,\n             \"checkpoint\": str(SOURCE), \"quantization\": \"iq4_nl\", \"decoder\": str(DECODER),\n             \"log\": str(LOG), \"stages\": {}, \"error\": None}\n    save_status()\n    try:\n        if REPO.exists() and not (REPO / \".git\").exists():\n            raise RuntimeError(f\"Refusing to overwrite non-git path: {REPO}\")\n        if not REPO.exists():\n            run_stage(\"clone\", [\"git\", \"clone\", \"--depth\", \"1\", \"https://github.com/NVIDIA/NeMo-Speech.cpp.git\", str(REPO)])\n        head = subprocess.run([\"git\", \"-C\", str(REPO), \"rev-parse\", \"HEAD\"], check=True, capture_output=True, text=True).stdout.strip()\n        if head != REVISION:\n            dirty = subprocess.run([\"git\", \"-C\", str(REPO), \"diff\", \"--quiet\"], check=False).returncode\n            if dirty:\n                raise RuntimeError(f\"Refusing to checkout {REVISION}; existing tracked changes would be lost.\")\n            run_stage(\"fetch_revision\", [\"git\", \"-C\", str(REPO), \"fetch\", \"--depth\", \"1\", \"origin\", REVISION])\n            run_stage(\"checkout_revision\", [\"git\", \"-C\", str(REPO), \"checkout\", \"--detach\", REVISION])\n        run_stage(\"submodule\", [\"git\", \"-C\", str(REPO), \"submodule\", \"update\", \"--init\", \"--depth\", \"1\", \"llama.cpp\"])\n        run_stage(\"system_dependencies\", [\"apt-get\", \"install\", \"-y\", \"build-essential\", \"cmake\", \"ninja-build\", \"git\", \"pkg-config\", \"libsentencepiece-dev\"])\n        if not shutil.which(\"nvcc\"):\n            raise RuntimeError(\"nvcc is missing; the CUDA toolkit must be available in the Colab runtime.\")\n        if not SOURCE.is_dir():\n            raise FileNotFoundError(f\"Expected existing BF16 snapshot; refusing model download: {SOURCE}\")\n        torch_version = importlib.metadata.version(\"torch\")\n        if torch_version != \"2.11.0+cu130\":\n            raise RuntimeError(f\"Expected existing torch 2.11.0+cu130; found {torch_version}. No pip install was attempted.\")\n        torchaudio_version = importlib.metadata.version(\"torchaudio\")\n        requirements = (REPO / \"requirements.txt\").read_text(encoding=\"utf-8\").splitlines()\n        filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith(\"#\") and not line.startswith(\"--\")\n                    and not re.match(r\"^(torch|torchaudio)([<>=!~; ]|$)\", line.strip(), re.I)]\n        req_file = REPORT_DIR / \"cpp-requirements.txt\"\n        constraint_file = REPORT_DIR / \"torch-constraints.txt\"\n        req_file.write_text(\"\\n\".join(filtered) + \"\\n\", encoding=\"utf-8\")\n        constraint_file.write_text(f\"torch=={torch_version}\\ntorchaudio=={torchaudio_version}\\n\", encoding=\"utf-8\")\n        run_stage(\"converter_dependencies\", [sys.executable, \"-m\", \"pip\", \"install\", \"--disable-pip-version-check\",\n                  \"--upgrade-strategy\", \"only-if-needed\", \"--constraint\", str(constraint_file), \"-r\", str(req_file)], cwd=REPO)\n        if importlib.metadata.version(\"torch\") != torch_version:\n            raise RuntimeError(\"Converter dependency install changed the existing torch version.\")\n        patch_gpu_layers()\n        configure = [\"scripts/configure.sh\", \"cuda-asr\", \"-DGGML_CUDA=ON\", \"-DCMAKE_CUDA_ARCHITECTURES=75\",\n                     \"-DNEMO_SPEECH_GGML_PATCHED=OFF\", \"-DNEMO_SPEECH_BUILD_ASR=ON\", \"-DNEMO_SPEECH_BUILD_SPEECHLM=ON\",\n                     \"-DNEMO_SPEECH_BUILD_DIAR=OFF\", \"-DNEMO_SPEECH_BUILD_TTS=OFF\", \"-DNEMO_SPEECH_BUILD_NMT=OFF\",\n                     \"-DNEMO_SPEECH_BUILD_CLI=ON\", \"-DNEMO_SPEECH_BUILD_MIC_CAPTURE=OFF\", \"-DNEMO_SPEECH_BUILD_HTTP=OFF\",\n                     \"-DNEMO_SPEECH_BUILD_GRPC=OFF\", \"-DNEMO_SPEECH_BUILD_S2S=OFF\"]\n        run_stage(\"configure\", configure, cwd=REPO)\n        run_stage(\"build\", [\"cmake\", \"--build\", \"--preset\", \"cuda-asr\", \"--parallel\", \"4\"], cwd=REPO)\n        run_stage(\"convert_iq4_nl\", [sys.executable, str(REPO / \"convert_model.py\"), str(SOURCE), \"--architecture\", \"salm\",\n                  \"--outtype\", \"iq4_nl\", \"--outfile\", str(DECODER)], cwd=REPO)\n        encoders = list(OUT.glob(\"*-Encoder-F16.gguf\"))\n        if not DECODER.is_file() or not encoders:\n            raise RuntimeError(\"Converter exited successfully but decoder or F16 speech encoder is missing.\")\n        state.update({\"status\": \"complete\", \"phase\": \"complete\", \"finished_at_utc\": utc(),\n                      \"encoder_files\": [str(path) for path in encoders]})\n        save_status()\n        print(f\"[prepare_cpp] complete: {DECODER}\", flush=True)\n        return 0\n    except Exception as exc:\n        state.update({\"status\": \"failed\", \"phase\": state.get(\"phase\", \"unknown\"), \"finished_at_utc\": utc(),\n                      \"error\": {\"type\": type(exc).__name__, \"message\": str(exc)}})\n        save_status()\n        print(f\"[prepare_cpp] failed in {state['phase']}: {type(exc).__name__}: {exc}\", file=sys.stderr, flush=True)\n        raise\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"''')
root = pathlib.Path('/content/nemotron-transcribe')
(root / 'prepare_cpp.py').write_text(source)
log = (root / 'cpp-launch.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(root / 'prepare_cpp.py')], stdout=log, stderr=subprocess.STDOUT)
(root / 'cpp-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(root / 'prepare_cpp.py')}))
print('Started C++ build and conversion process', p.pid, flush=True)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (2492178851.py, line 2)

In [ ]:
import base64, json, pathlib, subprocess, sys
source = base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcmVwYXJlIHBpbm5lZCBOZU1vLVNwZWVjaC5jcHAsIGJ1aWxkIFQ0IENVREEgU0FMTSwgYW5kIGNvbnZlcnQgYSBsb2NhbCBjaGVja3BvaW50LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKClJFVklTSU9OID0gImQ4YTFhYmYyZWE5ZDExZThmOThhZDBlMTBjYzJjMTJlZjFjYjNhMTQiClJFUE8gPSBQYXRoKCIvY29udGVudC9uZW1vLXNwZWVjaCIpClNPVVJDRSA9IFBhdGgoIi9jb250ZW50L2hmLWNhY2hlL21vZGVscy0tbnZpZGlhLS1OVklESUEtTmVtb3Ryb24tMy41LVRyYW5zY3JpYmUtMzBCLUE0Qi9zbmFwc2hvdHMvYzE3NjY3NjQxNjQyYzQyNjVmZjJiNzAzN2RkMTAxYjdkNDU2NzYzNCIpCk9VVCA9IFBhdGgoIi9jb250ZW50L25lbW90cm9uLWdndWYiKQpSRVBPUlRfRElSID0gUGF0aCgiL2NvbnRlbnQvbmVtb3Ryb24tdHJhbnNjcmliZSIpClNUQVRVUyA9IFJFUE9SVF9ESVIgLyAiY3BwLXN0YXR1cy5qc29uIgpMT0cgPSBSRVBPUlRfRElSIC8gInByZXBhcmVfY3BwLmxvZyIKUEFUQ0hfRklMRSA9IFJFUE9SVF9ESVIgLyAibl9ncHVfbGF5ZXJzLnBhdGNoIgpERUNPREVSID0gT1VUIC8gImRlY29kZXItSVE0X05MLmdndWYiCgoKZGVmIHV0YygpIC0+IHN0cjoKICAgIHJldHVybiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQoKQoKCmRlZiBzYXZlX3N0YXR1cygpIC0+IE5vbmU6CiAgICBSRVBPUlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIFNUQVRVUy5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCB0ZW1wZmlsZS5OYW1lZFRlbXBvcmFyeUZpbGUoInciLCBlbmNvZGluZz0idXRmLTgiLCBkaXI9UkVQT1JUX0RJUiwgZGVsZXRlPUZhbHNlKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChzdGF0ZSwgZiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikKICAgICAgICBmLndyaXRlKCJcbiIpCiAgICAgICAgZi5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoZi5maWxlbm8oKSkKICAgICAgICB0bXAgPSBQYXRoKGYubmFtZSkKICAgIHRtcC5yZXBsYWNlKFNUQVRVUykKCgpkZWYgcnVuX3N0YWdlKG5hbWU6IHN0ciwgY29tbWFuZDogbGlzdFtzdHJdLCBjd2Q6IFBhdGggfCBOb25lID0gTm9uZSkgLT4gTm9uZToKICAgIHN0ZXAgPSB7InN0YXR1cyI6ICJydW5uaW5nIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCksICJjb21tYW5kIjogY29tbWFuZH0KICAgIHN0YXRlWyJwaGFzZSJdID0gbmFtZQogICAgc3RhdGVbInN0YWdlcyJdW25hbWVdID0gc3RlcAogICAgc2F2ZV9zdGF0dXMoKQogICAgcHJpbnQoZiJbcHJlcGFyZV9jcHBdIHN0YXJ0IHtuYW1lfTogeycgJy5qb2luKGNvbW1hbmQpfSIsIGZsdXNoPVRydWUpCiAgICB0cnk6CiAgICAgICAgd2l0aCBMT0cub3BlbigiYSIsIGVuY29kaW5nPSJ1dGYtOCIsIGJ1ZmZlcmluZz0xKSBhcyBsb2c6CiAgICAgICAgICAgIGxvZy53cml0ZShmIlxuW3t1dGMoKX1dIHtuYW1lfTogeycgJy5qb2luKGNvbW1hbmQpfVxuIikKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29tbWFuZCwgY3dkPWN3ZCwgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwgc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULCB0ZXh0PVRydWUsIGJ1ZnNpemU9MSkKICAgICAgICAgICAgYXNzZXJ0IHByb2Nlc3Muc3Rkb3V0IGlzIG5vdCBOb25lCiAgICAgICAgICAgIGZvciBsaW5lIGluIHByb2Nlc3Muc3Rkb3V0OgogICAgICAgICAgICAgICAgcHJpbnQobGluZSwgZW5kPSIiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgbG9nLndyaXRlKGxpbmUpCiAgICAgICAgICAgICAgICBsb2cuZmx1c2goKQogICAgICAgICAgICBjb2RlID0gcHJvY2Vzcy53YWl0KCkKICAgICAgICBpZiBjb2RlOgogICAgICAgICAgICByYWlzZSBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcihjb2RlLCBjb21tYW5kKQogICAgICAgIHN0ZXBbInN0YXR1cyJdID0gImNvbXBsZXRlIgogICAgICAgIHN0ZXBbImZpbmlzaGVkX2F0X3V0YyJdID0gdXRjKCkKICAgICAgICBzYXZlX3N0YXR1cygpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzdGVwLnVwZGF0ZSh7InN0YXR1cyI6ICJmYWlsZWQiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksICJlcnJvciI6IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9In0pCiAgICAgICAgc2F2ZV9zdGF0dXMoKQogICAgICAgIHJhaXNlCgoKZGVmIHBhdGNoX2dwdV9sYXllcnMoKSAtPiBOb25lOgogICAgbmFtZSA9ICJwYXRjaF9ncHVfbGF5ZXJzIgogICAgc3RhdGVbInBoYXNlIl0gPSBuYW1lCiAgICBzdGF0ZVsic3RhZ2VzIl1bbmFtZV0gPSB7InN0YXR1cyI6ICJydW5uaW5nIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCl9CiAgICBzYXZlX3N0YXR1cygpCiAgICBwYXRoID0gUkVQTyAvICJzcmMvc3BlZWNobG0vc2FsbV9lbmdpbmUuY3BwIgogICAgc291cmNlID0gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKICAgIG9sZCA9ICIgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7IgogICAgbmV3ID0gJycnICAgICAgICBjb25zdCBjaGFyKiBncHVfbGF5ZXJzX2VudiA9IHN0ZDo6Z2V0ZW52KCJORU1PVFJPTl9HUFVfTEFZRVJTIik7CiAgICAgICAgaWYgKGdwdV9sYXllcnNfZW52KSB7CiAgICAgICAgICAgIGNoYXIqIHBhcnNlZF9lbmQgPSBudWxscHRyOwogICAgICAgICAgICBjb25zdCBsb25nIGdwdV9sYXllcnMgPSBzdGQ6OnN0cnRvbChncHVfbGF5ZXJzX2VudiwgJnBhcnNlZF9lbmQsIDEwKTsKICAgICAgICAgICAgaWYgKHBhcnNlZF9lbmQgPT0gZ3B1X2xheWVyc19lbnYgfHwgKnBhcnNlZF9lbmQgIT0gJ1xcMCcgfHwgZ3B1X2xheWVycyA8IDAgfHwgZ3B1X2xheWVycyA+IDk5OSkKICAgICAgICAgICAgICAgIHRocm93IHN0ZDo6aW52YWxpZF9hcmd1bWVudCgiTkVNT1RST05fR1BVX0xBWUVSUyBtdXN0IGJlIGFuIGludGVnZXIgZnJvbSAwIHRvIDk5OSIpOwogICAgICAgICAgICBtcC5uX2dwdV9sYXllcnMgPSBzdGF0aWNfY2FzdDxpbnQ+KGdwdV9sYXllcnMpOwogICAgICAgIH0gZWxzZSB7CiAgICAgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7CiAgICAgICAgfScnJwogICAgaWYgIk5FTU9UUk9OX0dQVV9MQVlFUlMiIG5vdCBpbiBzb3VyY2U6CiAgICAgICAgaWYgc291cmNlLmNvdW50KG9sZCkgIT0gMToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJQaW5uZWQgU0FMTSBzb3VyY2Ugbm8gbG9uZ2VyIGhhcyBleGFjdGx5IG9uZSBleHBlY3RlZCBuX2dwdV9sYXllcnMgYXNzaWdubWVudC4iKQogICAgICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgoIi5jcHAudG1wIikKICAgICAgICB0bXAud3JpdGVfdGV4dChzb3VyY2UucmVwbGFjZShvbGQsIG5ldywgMSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgdG1wLnJlcGxhY2UocGF0aCkKICAgIGVsaWYgIm1wLm5fZ3B1X2xheWVycyA9IHN0YXRpY19jYXN0PGludD4oZ3B1X2xheWVycyk7IiBub3QgaW4gc291cmNlOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRXhpc3RpbmcgTkVNT1RST05fR1BVX0xBWUVSUyBwYXRjaCBkb2VzIG5vdCBtYXRjaCB0aGUgZXhwZWN0ZWQgaW1wbGVtZW50YXRpb24uIikKICAgIGRpZmYgPSBzdWJwcm9jZXNzLnJ1bihbImdpdCIsICJkaWZmIiwgIi0tYmluYXJ5IiwgIi0tIiwgInNyYy9zcGVlY2hsbS9zYWxtX2VuZ2luZS5jcHAiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICBjd2Q9UkVQTywgY2hlY2s9VHJ1ZSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKS5zdGRvdXQKICAgIGlmIG5vdCBkaWZmOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTm8gc291cmNlIGRpZmYgd2FzIHByb2R1Y2VkIGZvciB0aGUgR1BVLWxheWVyIG92ZXJyaWRlLiIpCiAgICBQQVRDSF9GSUxFLndyaXRlX3RleHQoZGlmZiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHN0YXRlWyJwYXRjaF9maWxlIl0gPSBzdHIoUEFUQ0hfRklMRSkKICAgIHN0YXRlWyJzdGFnZXMiXVtuYW1lXS51cGRhdGUoeyJzdGF0dXMiOiAiY29tcGxldGUiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksICJkaWZmX2J5dGVzIjogbGVuKGRpZmYuZW5jb2RlKCkpfSkKICAgIHNhdmVfc3RhdHVzKCkKICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBzYXZlZCBzb3VyY2UgZGlmZjoge1BBVENIX0ZJTEV9IiwgZmx1c2g9VHJ1ZSkKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIFJFUE9SVF9ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgT1VULm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGdsb2JhbCBzdGF0ZQogICAgc3RhdGUgPSB7InN0YXR1cyI6ICJydW5uaW5nIiwgInBoYXNlIjogInN0YXJ0dXAiLCAic3RhcnRlZF9hdF91dGMiOiB1dGMoKSwgInJldmlzaW9uIjogUkVWSVNJT04sCiAgICAgICAgICAgICAiY2hlY2twb2ludCI6IHN0cihTT1VSQ0UpLCAicXVhbnRpemF0aW9uIjogImlxNF9ubCIsICJkZWNvZGVyIjogc3RyKERFQ09ERVIpLAogICAgICAgICAgICAgImxvZyI6IHN0cihMT0cpLCAic3RhZ2VzIjoge30sICJlcnJvciI6IE5vbmV9CiAgICBzYXZlX3N0YXR1cygpCiAgICB0cnk6CiAgICAgICAgaWYgUkVQTy5leGlzdHMoKSBhbmQgbm90IChSRVBPIC8gIi5naXQiKS5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUmVmdXNpbmcgdG8gb3ZlcndyaXRlIG5vbi1naXQgcGF0aDoge1JFUE99IikKICAgICAgICBpZiBub3QgUkVQTy5leGlzdHMoKToKICAgICAgICAgICAgcnVuX3N0YWdlKCJjbG9uZSIsIFsiZ2l0IiwgImNsb25lIiwgIi0tZGVwdGgiLCAiMSIsICJodHRwczovL2dpdGh1Yi5jb20vTlZJRElBL05lTW8tU3BlZWNoLmNwcC5naXQiLCBzdHIoUkVQTyldKQogICAgICAgIGhlYWQgPSBzdWJwcm9jZXNzLnJ1bihbImdpdCIsICItQyIsIHN0cihSRVBPKSwgInJldi1wYXJzZSIsICJIRUFEIl0sIGNoZWNrPVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkuc3Rkb3V0LnN0cmlwKCkKICAgICAgICBpZiBoZWFkICE9IFJFVklTSU9OOgogICAgICAgICAgICBkaXJ0eSA9IHN1YnByb2Nlc3MucnVuKFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZGlmZiIsICItLXF1aWV0Il0sIGNoZWNrPUZhbHNlKS5yZXR1cm5jb2RlCiAgICAgICAgICAgIGlmIGRpcnR5OgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUmVmdXNpbmcgdG8gY2hlY2tvdXQge1JFVklTSU9OfTsgZXhpc3RpbmcgdHJhY2tlZCBjaGFuZ2VzIHdvdWxkIGJlIGxvc3QuIikKICAgICAgICAgICAgcnVuX3N0YWdlKCJmZXRjaF9yZXZpc2lvbiIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZmV0Y2giLCAiLS1kZXB0aCIsICIxIiwgIm9yaWdpbiIsIFJFVklTSU9OXSkKICAgICAgICAgICAgcnVuX3N0YWdlKCJjaGVja291dF9yZXZpc2lvbiIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiY2hlY2tvdXQiLCAiLS1kZXRhY2giLCBSRVZJU0lPTl0pCiAgICAgICAgcnVuX3N0YWdlKCJzdWJtb2R1bGUiLCBbImdpdCIsICItQyIsIHN0cihSRVBPKSwgInN1Ym1vZHVsZSIsICJ1cGRhdGUiLCAiLS1pbml0IiwgIi0tZGVwdGgiLCAiMSIsICJsbGFtYS5jcHAiXSkKICAgICAgICBydW5fc3RhZ2UoInN5c3RlbV9kZXBlbmRlbmNpZXMiLCBbImFwdC1nZXQiLCAiaW5zdGFsbCIsICIteSIsICJidWlsZC1lc3NlbnRpYWwiLCAiY21ha2UiLCAibmluamEtYnVpbGQiLCAiZ2l0IiwgInBrZy1jb25maWciLCAibGlic2VudGVuY2VwaWVjZS1kZXYiXSkKICAgICAgICBpZiBub3Qgc2h1dGlsLndoaWNoKCJudmNjIik6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigibnZjYyBpcyBtaXNzaW5nOyB0aGUgQ1VEQSB0b29sa2l0IG11c3QgYmUgYXZhaWxhYmxlIGluIHRoZSBDb2xhYiBydW50aW1lLiIpCiAgICAgICAgaWYgbm90IFNPVVJDRS5pc19kaXIoKToKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJFeHBlY3RlZCBleGlzdGluZyBCRjE2IHNuYXBzaG90OyByZWZ1c2luZyBtb2RlbCBkb3dubG9hZDoge1NPVVJDRX0iKQogICAgICAgIHRvcmNoX3ZlcnNpb24gPSBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbigidG9yY2giKQogICAgICAgIGlmIHRvcmNoX3ZlcnNpb24gIT0gIjIuMTEuMCtjdTEzMCI6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkV4cGVjdGVkIGV4aXN0aW5nIHRvcmNoIDIuMTEuMCtjdTEzMDsgZm91bmQge3RvcmNoX3ZlcnNpb259LiBObyBwaXAgaW5zdGFsbCB3YXMgYXR0ZW1wdGVkLiIpCiAgICAgICAgdG9yY2hhdWRpb192ZXJzaW9uID0gaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24oInRvcmNoYXVkaW8iKQogICAgICAgIHJlcXVpcmVtZW50cyA9IChSRVBPIC8gInJlcXVpcmVtZW50cy50eHQiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpCiAgICAgICAgZmlsdGVyZWQgPSBbbGluZSBmb3IgbGluZSBpbiByZXF1aXJlbWVudHMgaWYgbGluZS5zdHJpcCgpIGFuZCBub3QgbGluZS5sc3RyaXAoKS5zdGFydHN3aXRoKCIjIikgYW5kIG5vdCBsaW5lLnN0YXJ0c3dpdGgoIi0tIikKICAgICAgICAgICAgICAgICAgICBhbmQgbm90IHJlLm1hdGNoKHIiXih0b3JjaHx0b3JjaGF1ZGlvKShbPD49IX47IF18JCkiLCBsaW5lLnN0cmlwKCksIHJlLkkpXQogICAgICAgIHJlcV9maWxlID0gUkVQT1JUX0RJUiAvICJjcHAtcmVxdWlyZW1lbnRzLnR4dCIKICAgICAgICBjb25zdHJhaW50X2ZpbGUgPSBSRVBPUlRfRElSIC8gInRvcmNoLWNvbnN0cmFpbnRzLnR4dCIKICAgICAgICByZXFfZmlsZS53cml0ZV90ZXh0KCJcbiIuam9pbihmaWx0ZXJlZCkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIGNvbnN0cmFpbnRfZmlsZS53cml0ZV90ZXh0KGYidG9yY2g9PXt0b3JjaF92ZXJzaW9ufVxudG9yY2hhdWRpbz09e3RvcmNoYXVkaW9fdmVyc2lvbn1cbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgcnVuX3N0YWdlKCJjb252ZXJ0ZXJfZGVwZW5kZW5jaWVzIiwgW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLS1kaXNhYmxlLXBpcC12ZXJzaW9uLWNoZWNrIiwKICAgICAgICAgICAgICAgICAgIi0tdXBncmFkZS1zdHJhdGVneSIsICJvbmx5LWlmLW5lZWRlZCIsICItLWNvbnN0cmFpbnQiLCBzdHIoY29uc3RyYWludF9maWxlKSwgIi1yIiwgc3RyKHJlcV9maWxlKV0sIGN3ZD1SRVBPKQogICAgICAgIGlmIGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCJ0b3JjaCIpICE9IHRvcmNoX3ZlcnNpb246CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ29udmVydGVyIGRlcGVuZGVuY3kgaW5zdGFsbCBjaGFuZ2VkIHRoZSBleGlzdGluZyB0b3JjaCB2ZXJzaW9uLiIpCiAgICAgICAgcGF0Y2hfZ3B1X2xheWVycygpCiAgICAgICAgY29uZmlndXJlID0gWyJzY3JpcHRzL2NvbmZpZ3VyZS5zaCIsICJjdWRhLWFzciIsICItREdHTUxfQ1VEQT1PTiIsICItRENNQUtFX0NVREFfQVJDSElURUNUVVJFUz03NSIsCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0dHTUxfUEFUQ0hFRD1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9BU1I9T04iLCAiLURORU1PX1NQRUVDSF9CVUlMRF9TUEVFQ0hMTT1PTiIsCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0RJQVI9T0ZGIiwgIi1ETkVNT19TUEVFQ0hfQlVJTERfVFRTPU9GRiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX05NVD1PRkYiLAogICAgICAgICAgICAgICAgICAgICAiLURORU1PX1NQRUVDSF9CVUlMRF9DTEk9T04iLCAiLURORU1PX1NQRUVDSF9CVUlMRF9NSUNfQ0FQVFVSRT1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9IVFRQPU9GRiIsCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0dSUEM9T0ZGIiwgIi1ETkVNT19TUEVFQ0hfQlVJTERfUzJTPU9GRiJdCiAgICAgICAgcnVuX3N0YWdlKCJjb25maWd1cmUiLCBjb25maWd1cmUsIGN3ZD1SRVBPKQogICAgICAgIHJ1bl9zdGFnZSgiYnVpbGQiLCBbImNtYWtlIiwgIi0tYnVpbGQiLCAiLS1wcmVzZXQiLCAiY3VkYS1hc3IiLCAiLS1wYXJhbGxlbCIsICI0Il0sIGN3ZD1SRVBPKQogICAgICAgIHJ1bl9zdGFnZSgiY29udmVydF9pcTRfbmwiLCBbc3lzLmV4ZWN1dGFibGUsIHN0cihSRVBPIC8gImNvbnZlcnRfbW9kZWwucHkiKSwgc3RyKFNPVVJDRSksICItLWFyY2hpdGVjdHVyZSIsICJzYWxtIiwKICAgICAgICAgICAgICAgICAgIi0tb3V0dHlwZSIsICJpcTRfbmwiLCAiLS1vdXRmaWxlIiwgc3RyKERFQ09ERVIpXSwgY3dkPVJFUE8pCiAgICAgICAgZW5jb2RlcnMgPSBsaXN0KE9VVC5nbG9iKCIqLUVuY29kZXItRjE2LmdndWYiKSkKICAgICAgICBpZiBub3QgREVDT0RFUi5pc19maWxlKCkgb3Igbm90IGVuY29kZXJzOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvbnZlcnRlciBleGl0ZWQgc3VjY2Vzc2Z1bGx5IGJ1dCBkZWNvZGVyIG9yIEYxNiBzcGVlY2ggZW5jb2RlciBpcyBtaXNzaW5nLiIpCiAgICAgICAgc3RhdGUudXBkYXRlKHsic3RhdHVzIjogImNvbXBsZXRlIiwgInBoYXNlIjogImNvbXBsZXRlIiwgImZpbmlzaGVkX2F0X3V0YyI6IHV0YygpLAogICAgICAgICAgICAgICAgICAgICAgImVuY29kZXJfZmlsZXMiOiBbc3RyKHBhdGgpIGZvciBwYXRoIGluIGVuY29kZXJzXX0pCiAgICAgICAgc2F2ZV9zdGF0dXMoKQogICAgICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBjb21wbGV0ZToge0RFQ09ERVJ9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gMAogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgc3RhdGUudXBkYXRlKHsic3RhdHVzIjogImZhaWxlZCIsICJwaGFzZSI6IHN0YXRlLmdldCgicGhhc2UiLCAidW5rbm93biIpLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksCiAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiB7InR5cGUiOiB0eXBlKGV4YykuX19uYW1lX18sICJtZXNzYWdlIjogc3RyKGV4Yyl9fSkKICAgICAgICBzYXZlX3N0YXR1cygpCiAgICAgICAgcHJpbnQoZiJbcHJlcGFyZV9jcHBdIGZhaWxlZCBpbiB7c3RhdGVbJ3BoYXNlJ119OiB7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iLCBmaWxlPXN5cy5zdGRlcnIsIGZsdXNoPVRydWUpCiAgICAgICAgcmFpc2UKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==').decode('utf-8')
root = pathlib.Path('/content/nemotron-transcribe')
(root / 'prepare_cpp.py').write_text(source)
log = (root / 'cpp-launch.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(root / 'prepare_cpp.py')], stdout=log, stderr=subprocess.STDOUT)
(root / 'cpp-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(root / 'prepare_cpp.py')}))
print('Started C++ build and conversion process', p.pid, flush=True)


Started C++ build and conversion process 6637


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "converter_dependencies", "started_at_utc": "2026-10-30T05:42:52.157329+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null}


prepare_cpp.log 
[2026-10-30T05:42:52.167956+00:00] submodule: git -C /content/nemo-speech submodule update --init --depth 1 llama.cpp

[2026-10-30T05:42:52.201387+00:00] system_dependencies: apt-get install -y build-essential cmake ninja-build git pkg-config libsentencepiece-dev
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1build7).
ninja-build is already the newest version (1.11.1-2).
git is already the newest version (1:2.43.0-1ubuntu7.4).
pkg-config is already the newest version (1.8.1-2build1).
libsentencepiece-dev is already the newest version (0.2.0-1build1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.

[2026-10-30T05:42:54.454471+00:00] converter_dependencies: /usr/bin/python3 -m pip install --disable-pip-version-check --upgrade-strategy only-if-needed --constraint /content/nemotron-transcribe/torch-constraints.txt -r

cpp-launch.log [prepare_cpp] start submodule: git -C /content/nemo-speech submodule update --init --depth 1 llama.cpp
[prepare_cpp] start system_dependencies: apt-get install -y build-essential cmake ninja-build git pkg-config libsentencepiece-dev
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1build7).
ninja-build is already the newest version (1.11.1-2).
git is already the newest version (1:2.43.0-1ubuntu7.4).
pkg-config is already the newest version (1.8.1-2build1).
libsentencepiece-dev is already the newest version (0.2.0-1build1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.
[prepare_cpp] start converter_dependencies: /usr/bin/python3 -m pip install --disable-pip-version-check --upgrade-strategy only-if-needed --constraint /content/nemotron-transcribe/torch-constraints.txt -r /content/nemotron-transcribe/cpp-requirements.tx

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "converter_dependencies", "started_at_utc": "2026-10-30T05:42:52.157329+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null}


prepare_cpp.log 
[2026-10-30T05:42:52.167956+00:00] submodule: git -C /content/nemo-speech submodule update --init --depth 1 llama.cpp

[2026-10-30T05:42:52.201387+00:00] system_dependencies: apt-get install -y build-essential cmake ninja-build git pkg-config libsentencepiece-dev
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1build7).
ninja-build is already the newest version (1.11.1-2).
git is already the newest version (1:2.43.0-1ubuntu7.4).
pkg-config is already the newest version (1.8.1-2build1).
libsentencepiece-dev is already the newest version (0.2.0-1build1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.

[2026-10-30T05:42:54.454471+00:00] converter_dependencies: /usr/bin/python3 -m pip install --disable-pip-version-check --upgrade-strategy only-if-needed --constraint /content/nemotron-transcribe/torch-constraints.txt -r

cpp-launch.log [prepare_cpp] start submodule: git -C /content/nemo-speech submodule update --init --depth 1 llama.cpp
[prepare_cpp] start system_dependencies: apt-get install -y build-essential cmake ninja-build git pkg-config libsentencepiece-dev
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1build7).
ninja-build is already the newest version (1.11.1-2).
git is already the newest version (1:2.43.0-1ubuntu7.4).
pkg-config is already the newest version (1.8.1-2build1).
libsentencepiece-dev is already the newest version (0.2.0-1build1).
0 upgraded, 0 newly installed, 0 to remove and 54 not upgraded.
[prepare_cpp] start converter_dependencies: /usr/bin/python3 -m pip install --disable-pip-version-check --upgrade-strategy only-if-needed --constraint /content/nemotron-transcribe/torch-constraints.txt -r /content/nemotron-transcribe/cpp-requirements.tx

In [ ]:
import json
from pathlib import Path
import shutil
import psutil

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'prepare_cpp.log', 'cpp-launch.log']:
    path = root / name
    if path.exists():
        shutil.copyfile(path, root / ('dependency-attempt-1-' + name))
pid = json.loads((root / 'cpp-process.json').read_text())['pid']
try:
    process = psutil.Process(pid)
    assert 'prepare_cpp.py' in ' '.join(process.cmdline())
    targets = process.children(recursive=True) + [process]
    for target in reversed(targets):
        try:
            target.terminate()
        except psutil.NoSuchProcess:
            pass
    _, alive = psutil.wait_procs(targets, timeout=5)
    for target in alive:
        target.kill()
except psutil.NoSuchProcess:
    pass
print('Stopped only this experiment C++ preparation worker and its children.', flush=True)


Stopped only this experiment C++ preparation worker and its children.


In [ ]:
import base64, json, pathlib, subprocess, sys
root = pathlib.Path('/content/nemotron-transcribe')
source = base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiUHJlcGFyZSBwaW5uZWQgTmVNby1TcGVlY2guY3BwLCBidWlsZCBUNCBDVURBIFNBTE0sIGFuZCBjb252ZXJ0IGEgbG9jYWwgY2hlY2twb2ludC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhDQppbXBvcnQganNvbg0KaW1wb3J0IG9zDQppbXBvcnQgcmUNCmltcG9ydCBzaHV0aWwNCmltcG9ydCBzdWJwcm9jZXNzDQppbXBvcnQgc3lzDQppbXBvcnQgdGVtcGZpbGUNCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQ0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCg0KUkVWSVNJT04gPSAiZDhhMWFiZjJlYTlkMTFlOGY5OGFkMGUxMGNjMmMxMmVmMWNiM2ExNCINClJFUE8gPSBQYXRoKCIvY29udGVudC9uZW1vLXNwZWVjaCIpDQpTT1VSQ0UgPSBQYXRoKCIvY29udGVudC9oZi1jYWNoZS9tb2RlbHMtLW52aWRpYS0tTlZJRElBLU5lbW90cm9uLTMuNS1UcmFuc2NyaWJlLTMwQi1BNEIvc25hcHNob3RzL2MxNzY2NzY0MTY0MmM0MjY1ZmYyYjcwMzdkZDEwMWI3ZDQ1Njc2MzQiKQ0KT1VUID0gUGF0aCgiL2NvbnRlbnQvbmVtb3Ryb24tZ2d1ZiIpDQpSRVBPUlRfRElSID0gUGF0aCgiL2NvbnRlbnQvbmVtb3Ryb24tdHJhbnNjcmliZSIpDQpTVEFUVVMgPSBSRVBPUlRfRElSIC8gImNwcC1zdGF0dXMuanNvbiINCkxPRyA9IFJFUE9SVF9ESVIgLyAicHJlcGFyZV9jcHAubG9nIg0KUEFUQ0hfRklMRSA9IFJFUE9SVF9ESVIgLyAibl9ncHVfbGF5ZXJzLnBhdGNoIg0KREVDT0RFUiA9IE9VVCAvICJkZWNvZGVyLUlRNF9OTC5nZ3VmIg0KDQoNCmRlZiB1dGMoKSAtPiBzdHI6DQogICAgcmV0dXJuIGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpDQoNCg0KZGVmIHNhdmVfc3RhdHVzKCkgLT4gTm9uZToNCiAgICBSRVBPUlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBTVEFUVVMucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3aXRoIHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZSgidyIsIGVuY29kaW5nPSJ1dGYtOCIsIGRpcj1SRVBPUlRfRElSLCBkZWxldGU9RmFsc2UpIGFzIGY6DQogICAgICAgIGpzb24uZHVtcChzdGF0ZSwgZiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikNCiAgICAgICAgZi53cml0ZSgiXG4iKQ0KICAgICAgICBmLmZsdXNoKCkNCiAgICAgICAgb3MuZnN5bmMoZi5maWxlbm8oKSkNCiAgICAgICAgdG1wID0gUGF0aChmLm5hbWUpDQogICAgdG1wLnJlcGxhY2UoU1RBVFVTKQ0KDQoNCmRlZiBydW5fc3RhZ2UobmFtZTogc3RyLCBjb21tYW5kOiBsaXN0W3N0cl0sIGN3ZDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIHN0ZXAgPSB7InN0YXR1cyI6ICJydW5uaW5nIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCksICJjb21tYW5kIjogY29tbWFuZH0NCiAgICBzdGF0ZVsicGhhc2UiXSA9IG5hbWUNCiAgICBzdGF0ZVsic3RhZ2VzIl1bbmFtZV0gPSBzdGVwDQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBzdGFydCB7bmFtZX06IHsnICcuam9pbihjb21tYW5kKX0iLCBmbHVzaD1UcnVlKQ0KICAgIHRyeToNCiAgICAgICAgd2l0aCBMT0cub3BlbigiYSIsIGVuY29kaW5nPSJ1dGYtOCIsIGJ1ZmZlcmluZz0xKSBhcyBsb2c6DQogICAgICAgICAgICBsb2cud3JpdGUoZiJcblt7dXRjKCl9XSB7bmFtZX06IHsnICcuam9pbihjb21tYW5kKX1cbiIpDQogICAgICAgICAgICBwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBjd2Q9Y3dkLCBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsIHRleHQ9VHJ1ZSwgYnVmc2l6ZT0xKQ0KICAgICAgICAgICAgYXNzZXJ0IHByb2Nlc3Muc3Rkb3V0IGlzIG5vdCBOb25lDQogICAgICAgICAgICBmb3IgbGluZSBpbiBwcm9jZXNzLnN0ZG91dDoNCiAgICAgICAgICAgICAgICBwcmludChsaW5lLCBlbmQ9IiIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgbG9nLndyaXRlKGxpbmUpDQogICAgICAgICAgICAgICAgbG9nLmZsdXNoKCkNCiAgICAgICAgICAgIGNvZGUgPSBwcm9jZXNzLndhaXQoKQ0KICAgICAgICBpZiBjb2RlOg0KICAgICAgICAgICAgcmFpc2Ugc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IoY29kZSwgY29tbWFuZCkNCiAgICAgICAgc3RlcFsic3RhdHVzIl0gPSAiY29tcGxldGUiDQogICAgICAgIHN0ZXBbImZpbmlzaGVkX2F0X3V0YyJdID0gdXRjKCkNCiAgICAgICAgc2F2ZV9zdGF0dXMoKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBzdGVwLnVwZGF0ZSh7InN0YXR1cyI6ICJmYWlsZWQiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksICJlcnJvciI6IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9In0pDQogICAgICAgIHNhdmVfc3RhdHVzKCkNCiAgICAgICAgcmFpc2UNCg0KDQpkZWYgcGF0Y2hfZ3B1X2xheWVycygpIC0+IE5vbmU6DQogICAgbmFtZSA9ICJwYXRjaF9ncHVfbGF5ZXJzIg0KICAgIHN0YXRlWyJwaGFzZSJdID0gbmFtZQ0KICAgIHN0YXRlWyJzdGFnZXMiXVtuYW1lXSA9IHsic3RhdHVzIjogInJ1bm5pbmciLCAic3RhcnRlZF9hdF91dGMiOiB1dGMoKX0NCiAgICBzYXZlX3N0YXR1cygpDQogICAgcGF0aCA9IFJFUE8gLyAic3JjL3NwZWVjaGxtL3NhbG1fZW5naW5lLmNwcCINCiAgICBzb3VyY2UgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQ0KICAgIG9sZCA9ICIgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7Ig0KICAgIG5ldyA9ICcnJyAgICAgICAgY29uc3QgY2hhciogZ3B1X2xheWVyc19lbnYgPSBzdGQ6OmdldGVudigiTkVNT1RST05fR1BVX0xBWUVSUyIpOw0KICAgICAgICBpZiAoZ3B1X2xheWVyc19lbnYpIHsNCiAgICAgICAgICAgIGNoYXIqIHBhcnNlZF9lbmQgPSBudWxscHRyOw0KICAgICAgICAgICAgY29uc3QgbG9uZyBncHVfbGF5ZXJzID0gc3RkOjpzdHJ0b2woZ3B1X2xheWVyc19lbnYsICZwYXJzZWRfZW5kLCAxMCk7DQogICAgICAgICAgICBpZiAocGFyc2VkX2VuZCA9PSBncHVfbGF5ZXJzX2VudiB8fCAqcGFyc2VkX2VuZCAhPSAnXFwwJyB8fCBncHVfbGF5ZXJzIDwgMCB8fCBncHVfbGF5ZXJzID4gOTk5KQ0KICAgICAgICAgICAgICAgIHRocm93IHN0ZDo6aW52YWxpZF9hcmd1bWVudCgiTkVNT1RST05fR1BVX0xBWUVSUyBtdXN0IGJlIGFuIGludGVnZXIgZnJvbSAwIHRvIDk5OSIpOw0KICAgICAgICAgICAgbXAubl9ncHVfbGF5ZXJzID0gc3RhdGljX2Nhc3Q8aW50PihncHVfbGF5ZXJzKTsNCiAgICAgICAgfSBlbHNlIHsNCiAgICAgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7DQogICAgICAgIH0nJycNCiAgICBpZiAiTkVNT1RST05fR1BVX0xBWUVSUyIgbm90IGluIHNvdXJjZToNCiAgICAgICAgaWYgc291cmNlLmNvdW50KG9sZCkgIT0gMToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUGlubmVkIFNBTE0gc291cmNlIG5vIGxvbmdlciBoYXMgZXhhY3RseSBvbmUgZXhwZWN0ZWQgbl9ncHVfbGF5ZXJzIGFzc2lnbm1lbnQuIikNCiAgICAgICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeCgiLmNwcC50bXAiKQ0KICAgICAgICB0bXAud3JpdGVfdGV4dChzb3VyY2UucmVwbGFjZShvbGQsIG5ldywgMSksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgICAgIHRtcC5yZXBsYWNlKHBhdGgpDQogICAgZWxpZiAibXAubl9ncHVfbGF5ZXJzID0gc3RhdGljX2Nhc3Q8aW50PihncHVfbGF5ZXJzKTsiIG5vdCBpbiBzb3VyY2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRXhpc3RpbmcgTkVNT1RST05fR1BVX0xBWUVSUyBwYXRjaCBkb2VzIG5vdCBtYXRjaCB0aGUgZXhwZWN0ZWQgaW1wbGVtZW50YXRpb24uIikNCiAgICBkaWZmID0gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiZGlmZiIsICItLWJpbmFyeSIsICItLSIsICJzcmMvc3BlZWNobG0vc2FsbV9lbmdpbmUuY3BwIl0sDQogICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1SRVBPLCBjaGVjaz1UcnVlLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dA0KICAgIGlmIG5vdCBkaWZmOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHNvdXJjZSBkaWZmIHdhcyBwcm9kdWNlZCBmb3IgdGhlIEdQVS1sYXllciBvdmVycmlkZS4iKQ0KICAgIFBBVENIX0ZJTEUud3JpdGVfdGV4dChkaWZmLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgIHN0YXRlWyJwYXRjaF9maWxlIl0gPSBzdHIoUEFUQ0hfRklMRSkNCiAgICBzdGF0ZVsic3RhZ2VzIl1bbmFtZV0udXBkYXRlKHsic3RhdHVzIjogImNvbXBsZXRlIiwgImZpbmlzaGVkX2F0X3V0YyI6IHV0YygpLCAiZGlmZl9ieXRlcyI6IGxlbihkaWZmLmVuY29kZSgpKX0pDQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBzYXZlZCBzb3VyY2UgZGlmZjoge1BBVENIX0ZJTEV9IiwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgbWFpbigpIC0+IGludDoNCiAgICBSRVBPUlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBPVVQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIGdsb2JhbCBzdGF0ZQ0KICAgIHN0YXRlID0geyJzdGF0dXMiOiAicnVubmluZyIsICJwaGFzZSI6ICJzdGFydHVwIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCksICJyZXZpc2lvbiI6IFJFVklTSU9OLA0KICAgICAgICAgICAgICJjaGVja3BvaW50Ijogc3RyKFNPVVJDRSksICJxdWFudGl6YXRpb24iOiAiaXE0X25sIiwgImRlY29kZXIiOiBzdHIoREVDT0RFUiksDQogICAgICAgICAgICAgImxvZyI6IHN0cihMT0cpLCAic3RhZ2VzIjoge30sICJlcnJvciI6IE5vbmV9DQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHRyeToNCiAgICAgICAgaWYgUkVQTy5leGlzdHMoKSBhbmQgbm90IChSRVBPIC8gIi5naXQiKS5leGlzdHMoKToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSBub24tZ2l0IHBhdGg6IHtSRVBPfSIpDQogICAgICAgIGlmIG5vdCBSRVBPLmV4aXN0cygpOg0KICAgICAgICAgICAgcnVuX3N0YWdlKCJjbG9uZSIsIFsiZ2l0IiwgImNsb25lIiwgIi0tZGVwdGgiLCAiMSIsICJodHRwczovL2dpdGh1Yi5jb20vTlZJRElBL05lTW8tU3BlZWNoLmNwcC5naXQiLCBzdHIoUkVQTyldKQ0KICAgICAgICBoZWFkID0gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiLUMiLCBzdHIoUkVQTyksICJyZXYtcGFyc2UiLCAiSEVBRCJdLCBjaGVjaz1UcnVlLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dC5zdHJpcCgpDQogICAgICAgIGlmIGhlYWQgIT0gUkVWSVNJT046DQogICAgICAgICAgICBkaXJ0eSA9IHN1YnByb2Nlc3MucnVuKFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZGlmZiIsICItLXF1aWV0Il0sIGNoZWNrPUZhbHNlKS5yZXR1cm5jb2RlDQogICAgICAgICAgICBpZiBkaXJ0eToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJSZWZ1c2luZyB0byBjaGVja291dCB7UkVWSVNJT059OyBleGlzdGluZyB0cmFja2VkIGNoYW5nZXMgd291bGQgYmUgbG9zdC4iKQ0KICAgICAgICAgICAgcnVuX3N0YWdlKCJmZXRjaF9yZXZpc2lvbiIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZmV0Y2giLCAiLS1kZXB0aCIsICIxIiwgIm9yaWdpbiIsIFJFVklTSU9OXSkNCiAgICAgICAgICAgIHJ1bl9zdGFnZSgiY2hlY2tvdXRfcmV2aXNpb24iLCBbImdpdCIsICItQyIsIHN0cihSRVBPKSwgImNoZWNrb3V0IiwgIi0tZGV0YWNoIiwgUkVWSVNJT05dKQ0KICAgICAgICBydW5fc3RhZ2UoInN1Ym1vZHVsZSIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAic3VibW9kdWxlIiwgInVwZGF0ZSIsICItLWluaXQiLCAiLS1kZXB0aCIsICIxIiwgImxsYW1hLmNwcCJdKQ0KICAgICAgICBydW5fc3RhZ2UoInN5c3RlbV9kZXBlbmRlbmNpZXMiLCBbImFwdC1nZXQiLCAiaW5zdGFsbCIsICIteSIsICJidWlsZC1lc3NlbnRpYWwiLCAiY21ha2UiLCAibmluamEtYnVpbGQiLCAiZ2l0IiwgInBrZy1jb25maWciLCAibGlic2VudGVuY2VwaWVjZS1kZXYiXSkNCiAgICAgICAgaWYgbm90IHNodXRpbC53aGljaCgibnZjYyIpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJudmNjIGlzIG1pc3Npbmc7IHRoZSBDVURBIHRvb2xraXQgbXVzdCBiZSBhdmFpbGFibGUgaW4gdGhlIENvbGFiIHJ1bnRpbWUuIikNCiAgICAgICAgaWYgbm90IFNPVVJDRS5pc19kaXIoKToNCiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRXhwZWN0ZWQgZXhpc3RpbmcgQkYxNiBzbmFwc2hvdDsgcmVmdXNpbmcgbW9kZWwgZG93bmxvYWQ6IHtTT1VSQ0V9IikNCiAgICAgICAgdG9yY2hfdmVyc2lvbiA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCJ0b3JjaCIpDQogICAgICAgIGlmIHRvcmNoX3ZlcnNpb24gIT0gIjIuMTEuMCtjdTEzMCI6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFeHBlY3RlZCBleGlzdGluZyB0b3JjaCAyLjExLjArY3UxMzA7IGZvdW5kIHt0b3JjaF92ZXJzaW9ufS4gTm8gcGlwIGluc3RhbGwgd2FzIGF0dGVtcHRlZC4iKQ0KICAgICAgICB0b3JjaGF1ZGlvX3ZlcnNpb24gPSBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbigidG9yY2hhdWRpbyIpDQogICAgICAgIHJlcXVpcmVtZW50cyA9IChSRVBPIC8gInJlcXVpcmVtZW50cy50eHQiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpDQogICAgICAgIGZpbHRlcmVkID0gWyJnZ3VmPj0wLjEwLjAiLCAic2VudGVuY2VwaWVjZT49MC4xLjk4LDwwLjMuMCIsICJQeVlBTUw+PTYsPDciXQ0KICAgICAgICByZXFfZmlsZSA9IFJFUE9SVF9ESVIgLyAiY3BwLXJlcXVpcmVtZW50cy50eHQiDQogICAgICAgIGNvbnN0cmFpbnRfZmlsZSA9IFJFUE9SVF9ESVIgLyAidG9yY2gtY29uc3RyYWludHMudHh0Ig0KICAgICAgICByZXFfZmlsZS53cml0ZV90ZXh0KCJcbiIuam9pbihmaWx0ZXJlZCkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgICAgICBjb25zdHJhaW50X2ZpbGUud3JpdGVfdGV4dChmInRvcmNoPT17dG9yY2hfdmVyc2lvbn1cbnRvcmNoYXVkaW89PXt0b3JjaGF1ZGlvX3ZlcnNpb259XG5udW1weT09e2ltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCdudW1weScpfVxudHJhbnNmb3JtZXJzPT17aW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24oJ3RyYW5zZm9ybWVycycpfVxuIiwgZW5jb2Rpbmc9InV0Zi04IikNCiAgICAgICAgcnVuX3N0YWdlKCJjb252ZXJ0ZXJfZGVwZW5kZW5jaWVzIiwgW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLS1kaXNhYmxlLXBpcC12ZXJzaW9uLWNoZWNrIiwNCiAgICAgICAgICAgICAgICAgICItLXVwZ3JhZGUtc3RyYXRlZ3kiLCAib25seS1pZi1uZWVkZWQiLCAiLS1jb25zdHJhaW50Iiwgc3RyKGNvbnN0cmFpbnRfZmlsZSksICItciIsIHN0cihyZXFfZmlsZSldLCBjd2Q9UkVQTykNCiAgICAgICAgaWYgaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24oInRvcmNoIikgIT0gdG9yY2hfdmVyc2lvbjoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ29udmVydGVyIGRlcGVuZGVuY3kgaW5zdGFsbCBjaGFuZ2VkIHRoZSBleGlzdGluZyB0b3JjaCB2ZXJzaW9uLiIpDQogICAgICAgIHBhdGNoX2dwdV9sYXllcnMoKQ0KICAgICAgICBjb25maWd1cmUgPSBbInNjcmlwdHMvY29uZmlndXJlLnNoIiwgImN1ZGEtYXNyIiwgIi1ER0dNTF9DVURBPU9OIiwgIi1EQ01BS0VfQ1VEQV9BUkNISVRFQ1RVUkVTPTc1IiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0dHTUxfUEFUQ0hFRD1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9BU1I9T04iLCAiLURORU1PX1NQRUVDSF9CVUlMRF9TUEVFQ0hMTT1PTiIsDQogICAgICAgICAgICAgICAgICAgICAiLURORU1PX1NQRUVDSF9CVUlMRF9ESUFSPU9GRiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX1RUUz1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9OTVQ9T0ZGIiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0NMST1PTiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX01JQ19DQVBUVVJFPU9GRiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX0hUVFA9T0ZGIiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0dSUEM9T0ZGIiwgIi1ETkVNT19TUEVFQ0hfQlVJTERfUzJTPU9GRiJdDQogICAgICAgIHJ1bl9zdGFnZSgiY29uZmlndXJlIiwgY29uZmlndXJlLCBjd2Q9UkVQTykNCiAgICAgICAgcnVuX3N0YWdlKCJidWlsZCIsIFsiY21ha2UiLCAiLS1idWlsZCIsICItLXByZXNldCIsICJjdWRhLWFzciIsICItLXBhcmFsbGVsIiwgIjQiXSwgY3dkPVJFUE8pDQogICAgICAgIHJ1bl9zdGFnZSgiY29udmVydF9pcTRfbmwiLCBbc3lzLmV4ZWN1dGFibGUsIHN0cihSRVBPIC8gImNvbnZlcnRfbW9kZWwucHkiKSwgc3RyKFNPVVJDRSksICItLWFyY2hpdGVjdHVyZSIsICJzYWxtIiwNCiAgICAgICAgICAgICAgICAgICItLW91dHR5cGUiLCAiaXE0X25sIiwgIi0tb3V0ZmlsZSIsIHN0cihERUNPREVSKV0sIGN3ZD1SRVBPKQ0KICAgICAgICBlbmNvZGVycyA9IGxpc3QoT1VULmdsb2IoIiotRW5jb2Rlci1GMTYuZ2d1ZiIpKQ0KICAgICAgICBpZiBub3QgREVDT0RFUi5pc19maWxlKCkgb3Igbm90IGVuY29kZXJzOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDb252ZXJ0ZXIgZXhpdGVkIHN1Y2Nlc3NmdWxseSBidXQgZGVjb2RlciBvciBGMTYgc3BlZWNoIGVuY29kZXIgaXMgbWlzc2luZy4iKQ0KICAgICAgICBzdGF0ZS51cGRhdGUoeyJzdGF0dXMiOiAiY29tcGxldGUiLCAicGhhc2UiOiAiY29tcGxldGUiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksDQogICAgICAgICAgICAgICAgICAgICAgImVuY29kZXJfZmlsZXMiOiBbc3RyKHBhdGgpIGZvciBwYXRoIGluIGVuY29kZXJzXX0pDQogICAgICAgIHNhdmVfc3RhdHVzKCkNCiAgICAgICAgcHJpbnQoZiJbcHJlcGFyZV9jcHBdIGNvbXBsZXRlOiB7REVDT0RFUn0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4gMA0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBzdGF0ZS51cGRhdGUoeyJzdGF0dXMiOiAiZmFpbGVkIiwgInBoYXNlIjogc3RhdGUuZ2V0KCJwaGFzZSIsICJ1bmtub3duIiksICJmaW5pc2hlZF9hdF91dGMiOiB1dGMoKSwNCiAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiB7InR5cGUiOiB0eXBlKGV4YykuX19uYW1lX18sICJtZXNzYWdlIjogc3RyKGV4Yyl9fSkNCiAgICAgICAgc2F2ZV9zdGF0dXMoKQ0KICAgICAgICBwcmludChmIltwcmVwYXJlX2NwcF0gZmFpbGVkIGluIHtzdGF0ZVsncGhhc2UnXX06IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIsIGZpbGU9c3lzLnN0ZGVyciwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmFpc2UNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K').decode('utf-8-sig')
worker = root / 'prepare_cpp.py'
worker.write_text(source)
log = (root / 'cpp-launch.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(worker)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
(root / 'cpp-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(worker)}))
print('Started', worker.name, 'PID', p.pid, flush=True)


Started prepare_cpp.py PID 10537


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "started_at_utc": "2026-10-30T05:45:54.739588+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null, "patch_file": "/content/nemotron-transcribe/n_gpu_layers.patch"}


prepare_cpp.log  GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.25.1
-- ggml commit:  bd4f514
-- llama.cpp version: 0.5.0-dev
CMAKE_BUILD_TYPE=Release
-- Configuring done (0.2s)
-- Generating done (0.1s)
-- Build files have been written to: /content/nemo-speech/build/cuda-asr

[2026-10-30T05:45:59.070105+00:00] build: cmake --build --preset cuda-asr --parallel 4
[0/2] Re-checking globbed directories...
[1/330] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/vec.cpp.o
[2/330] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/binary-ops.cpp.o
[3/330] Building CXX object ggml/src/CMakeFi

cpp-launch.log his warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.25.1
-- ggml commit:  bd4f514
-- llama.cpp version: 0.5.0-dev
CMAKE_BUILD_TYPE=Release
-- Configuring done (0.2s)
-- Generating done (0.1s)
-- Build files have been written to: /content/nemo-speech/build/cuda-asr
[prepare_cpp] start build: cmake --build --preset cuda-asr --parallel 4
[0/2] Re-checking globbed directories...
[1/330] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/vec.cpp.o
[2/330] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/binary-ops.cpp.o
[3/330] Building CXX object ggml/src/CMakeFil

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "started_at_utc": "2026-10-30T05:45:54.739588+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null, "patch_file": "/content/nemotron-transcribe/n_gpu_layers.patch"}


prepare_cpp.log bcast.cu.o
[27/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/cpy.cu.o
[28/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/diag.cu.o
[29/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/argsort.cu.o
[30/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/diagmask.cu.o
[31/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/dsv4-hc.cu.o
[32/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fattn-tile.cu.o
[33/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/cumsum.cu.o
[34/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fill.cu.o
[35/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fwht.cu.o
[36/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/gated_delta_net.cu.o
[37/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/gla.cu.o
[38/330] Building CUDA

cpp-launch.log bcast.cu.o
[27/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/cpy.cu.o
[28/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/diag.cu.o
[29/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/argsort.cu.o
[30/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/diagmask.cu.o
[31/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/dsv4-hc.cu.o
[32/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fattn-tile.cu.o
[33/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/cumsum.cu.o
[34/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fill.cu.o
[35/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/fwht.cu.o
[36/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/gated_delta_net.cu.o
[37/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/gla.cu.o
[38/330] Building CUDA 

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "started_at_utc": "2026-10-30T05:45:54.739588+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null, "patch_file": "/content/nemotron-transcribe/n_gpu_layers.patch"}


prepare_cpp.log src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad.cu.o
[53/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad_reflect_1d.cu.o
[54/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool1d.cu.o
[55/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool2d.cu.o
[56/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/roll.cu.o
[57/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/quantize.cu.o
[58/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/scale.cu.o
[59/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/rope.cu.o
[60/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/set-rows.cu.o
[61/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/set.cu.o
[62/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/snake.cu.o
[63/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir

cpp-launch.log src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad.cu.o
[53/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pad_reflect_1d.cu.o
[54/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool1d.cu.o
[55/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/pool2d.cu.o
[56/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/roll.cu.o
[57/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/quantize.cu.o
[58/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/scale.cu.o
[59/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/rope.cu.o
[60/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/set-rows.cu.o
[61/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/set.cu.o
[62/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/snake.cu.o
[63/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, '/content/nemo-speech/convert_model.py', '--help'], capture_output=True, text=True)
print('converter_help_returncode', result.returncode, flush=True)
print((result.stdout + result.stderr)[-3000:], flush=True)


converter_help_returncode 0


,f32}]
                        [--silero-version SILERO_VERSION]
                        [--from-whisper-ggml FROM_WHISPER_GGML]
                        [--salm-encoder {f16,none}]
                        [--salm-target SALM_TARGET] [--llama-cpp LLAMA_CPP]
                        [--llama-quantize LLAMA_QUANTIZE]
                        [--no-build-quantizer] [--force] [--dry-run]
                        source

Convert supported speech and translation checkpoints to runtime GGUF files.

positional arguments:
  source                local checkpoint/directory or Hugging Face repository
                        ID

options:
  -h, --help            show this help message and exit
  --outfile, -o OUTFILE
                        output GGUF path, or output artifact directory for S2S
  --architecture {auto,asr,diarization,pee,salm,dflash2,pnc,vad,tts,codec,nmt,s2s}
                        override checkpoint architecture detection
  --outtype OUTTYPE     output weight type; supported values 

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        print(name, json.dumps({key: value for key, value in state.items() if key != 'stages'}, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch.log', 'small-gpu-worker.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-2200:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "started_at_utc": "2026-10-30T05:45:54.739588+00:00", "revision": "d8a1abf2ea9d11e8f98ad0e10cc2c12ef1cb3a14", "checkpoint": "/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634", "quantization": "iq4_nl", "decoder": "/content/nemotron-gguf/decoder-IQ4_NL.gguf", "log": "/content/nemotron-transcribe/prepare_cpp.log", "error": null, "patch_file": "/content/nemotron-transcribe/n_gpu_layers.patch"}


prepare_cpp.log ir/tsembd.cu.o
[75/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/topk-moe.cu.o
[76/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/upscale.cu.o
[77/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/wkv.cu.o
[78/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/unary.cu.o
[79/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq192-dv128.cu.o
[80/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq112-dv112.cu.o
[81/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq320-dv256.cu.o
[82/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq128-dv128.cu.o
[83/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-til

cpp-launch.log ir/tsembd.cu.o
[75/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/topk-moe.cu.o
[76/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/upscale.cu.o
[77/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/wkv.cu.o
[78/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/unary.cu.o
[79/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq192-dv128.cu.o
[80/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq112-dv112.cu.o
[81/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq320-dv256.cu.o
[82/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile-instance-dkq128-dv128.cu.o
[83/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-tile

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "error": null}


prepare_cpp.log 1_16-ncols2_1.cu.o
[97/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_16-ncols2_2.cu.o
[98/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_1-ncols2_8.cu.o
[99/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_32-ncols2_2.cu.o
[100/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_4-ncols2_16.cu.o
[101/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_2-ncols2_4.cu.o
[102/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_2-ncols2_8.cu.o
[103/330] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_3

In [ ]:
import json
import os
import psutil

print(json.dumps({'cpus': os.cpu_count(), 'affinity': len(os.sched_getaffinity(0)), 'cpu_utilization': psutil.cpu_percent(interval=1, percpu=True), 'available_ram_gib': psutil.virtual_memory().available / 1024**3, 'load': os.getloadavg()}), flush=True)


{"cpus": 8, "affinity": 8, "cpu_utilization": [100.0, 6.9, 100.0, 100.0, 6.9, 100.0, 8.0, 5.9], "available_ram_gib": 48.34280014038086, "load": [4.13427734375, 3.83984375, 2.7724609375]}


In [ ]:
import os
os.environ['NEMOTRON_STOP_PREFIX'] = 'build-parallelism-4-'
import json
from pathlib import Path
import shutil
import psutil
import os

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'prepare_cpp.log', 'cpp-launch.log']:
    path = root / name
    if path.exists():
        shutil.copyfile(path, root / (os.environ.get('NEMOTRON_STOP_PREFIX', 'dependency-attempt-1-') + name))
pid = json.loads((root / 'cpp-process.json').read_text())['pid']
try:
    process = psutil.Process(pid)
    assert 'prepare_cpp.py' in ' '.join(process.cmdline())
    targets = process.children(recursive=True) + [process]
    for target in reversed(targets):
        try:
            target.terminate()
        except psutil.NoSuchProcess:
            pass
    _, alive = psutil.wait_procs(targets, timeout=5)
    for target in alive:
        target.kill()
except psutil.NoSuchProcess:
    pass
print('Stopped only this experiment C++ preparation worker and its children.', flush=True)


Stopped only this experiment C++ preparation worker and its children.


In [ ]:
import base64, json, pathlib, subprocess, sys
root = pathlib.Path('/content/nemotron-transcribe')
source = base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiUHJlcGFyZSBwaW5uZWQgTmVNby1TcGVlY2guY3BwLCBidWlsZCBUNCBDVURBIFNBTE0sIGFuZCBjb252ZXJ0IGEgbG9jYWwgY2hlY2twb2ludC4iIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhDQppbXBvcnQganNvbg0KaW1wb3J0IG9zDQppbXBvcnQgcmUNCmltcG9ydCBzaHV0aWwNCmltcG9ydCBzdWJwcm9jZXNzDQppbXBvcnQgc3lzDQppbXBvcnQgdGVtcGZpbGUNCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQ0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCg0KUkVWSVNJT04gPSAiZDhhMWFiZjJlYTlkMTFlOGY5OGFkMGUxMGNjMmMxMmVmMWNiM2ExNCINClJFUE8gPSBQYXRoKCIvY29udGVudC9uZW1vLXNwZWVjaCIpDQpTT1VSQ0UgPSBQYXRoKCIvY29udGVudC9oZi1jYWNoZS9tb2RlbHMtLW52aWRpYS0tTlZJRElBLU5lbW90cm9uLTMuNS1UcmFuc2NyaWJlLTMwQi1BNEIvc25hcHNob3RzL2MxNzY2NzY0MTY0MmM0MjY1ZmYyYjcwMzdkZDEwMWI3ZDQ1Njc2MzQiKQ0KT1VUID0gUGF0aCgiL2NvbnRlbnQvbmVtb3Ryb24tZ2d1ZiIpDQpSRVBPUlRfRElSID0gUGF0aCgiL2NvbnRlbnQvbmVtb3Ryb24tdHJhbnNjcmliZSIpDQpTVEFUVVMgPSBSRVBPUlRfRElSIC8gImNwcC1zdGF0dXMuanNvbiINCkxPRyA9IFJFUE9SVF9ESVIgLyAicHJlcGFyZV9jcHAubG9nIg0KUEFUQ0hfRklMRSA9IFJFUE9SVF9ESVIgLyAibl9ncHVfbGF5ZXJzLnBhdGNoIg0KREVDT0RFUiA9IE9VVCAvICJkZWNvZGVyLUlRNF9OTC5nZ3VmIg0KDQoNCmRlZiB1dGMoKSAtPiBzdHI6DQogICAgcmV0dXJuIGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpDQoNCg0KZGVmIHNhdmVfc3RhdHVzKCkgLT4gTm9uZToNCiAgICBSRVBPUlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBTVEFUVVMucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3aXRoIHRlbXBmaWxlLk5hbWVkVGVtcG9yYXJ5RmlsZSgidyIsIGVuY29kaW5nPSJ1dGYtOCIsIGRpcj1SRVBPUlRfRElSLCBkZWxldGU9RmFsc2UpIGFzIGY6DQogICAgICAgIGpzb24uZHVtcChzdGF0ZSwgZiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikNCiAgICAgICAgZi53cml0ZSgiXG4iKQ0KICAgICAgICBmLmZsdXNoKCkNCiAgICAgICAgb3MuZnN5bmMoZi5maWxlbm8oKSkNCiAgICAgICAgdG1wID0gUGF0aChmLm5hbWUpDQogICAgdG1wLnJlcGxhY2UoU1RBVFVTKQ0KDQoNCmRlZiBydW5fc3RhZ2UobmFtZTogc3RyLCBjb21tYW5kOiBsaXN0W3N0cl0sIGN3ZDogUGF0aCB8IE5vbmUgPSBOb25lKSAtPiBOb25lOg0KICAgIHN0ZXAgPSB7InN0YXR1cyI6ICJydW5uaW5nIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCksICJjb21tYW5kIjogY29tbWFuZH0NCiAgICBzdGF0ZVsicGhhc2UiXSA9IG5hbWUNCiAgICBzdGF0ZVsic3RhZ2VzIl1bbmFtZV0gPSBzdGVwDQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBzdGFydCB7bmFtZX06IHsnICcuam9pbihjb21tYW5kKX0iLCBmbHVzaD1UcnVlKQ0KICAgIHRyeToNCiAgICAgICAgd2l0aCBMT0cub3BlbigiYSIsIGVuY29kaW5nPSJ1dGYtOCIsIGJ1ZmZlcmluZz0xKSBhcyBsb2c6DQogICAgICAgICAgICBsb2cud3JpdGUoZiJcblt7dXRjKCl9XSB7bmFtZX06IHsnICcuam9pbihjb21tYW5kKX1cbiIpDQogICAgICAgICAgICBwcm9jZXNzID0gc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBjd2Q9Y3dkLCBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsIHRleHQ9VHJ1ZSwgYnVmc2l6ZT0xKQ0KICAgICAgICAgICAgYXNzZXJ0IHByb2Nlc3Muc3Rkb3V0IGlzIG5vdCBOb25lDQogICAgICAgICAgICBmb3IgbGluZSBpbiBwcm9jZXNzLnN0ZG91dDoNCiAgICAgICAgICAgICAgICBwcmludChsaW5lLCBlbmQ9IiIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgbG9nLndyaXRlKGxpbmUpDQogICAgICAgICAgICAgICAgbG9nLmZsdXNoKCkNCiAgICAgICAgICAgIGNvZGUgPSBwcm9jZXNzLndhaXQoKQ0KICAgICAgICBpZiBjb2RlOg0KICAgICAgICAgICAgcmFpc2Ugc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IoY29kZSwgY29tbWFuZCkNCiAgICAgICAgc3RlcFsic3RhdHVzIl0gPSAiY29tcGxldGUiDQogICAgICAgIHN0ZXBbImZpbmlzaGVkX2F0X3V0YyJdID0gdXRjKCkNCiAgICAgICAgc2F2ZV9zdGF0dXMoKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBzdGVwLnVwZGF0ZSh7InN0YXR1cyI6ICJmYWlsZWQiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksICJlcnJvciI6IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9In0pDQogICAgICAgIHNhdmVfc3RhdHVzKCkNCiAgICAgICAgcmFpc2UNCg0KDQpkZWYgcGF0Y2hfZ3B1X2xheWVycygpIC0+IE5vbmU6DQogICAgbmFtZSA9ICJwYXRjaF9ncHVfbGF5ZXJzIg0KICAgIHN0YXRlWyJwaGFzZSJdID0gbmFtZQ0KICAgIHN0YXRlWyJzdGFnZXMiXVtuYW1lXSA9IHsic3RhdHVzIjogInJ1bm5pbmciLCAic3RhcnRlZF9hdF91dGMiOiB1dGMoKX0NCiAgICBzYXZlX3N0YXR1cygpDQogICAgcGF0aCA9IFJFUE8gLyAic3JjL3NwZWVjaGxtL3NhbG1fZW5naW5lLmNwcCINCiAgICBzb3VyY2UgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQ0KICAgIG9sZCA9ICIgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7Ig0KICAgIG5ldyA9ICcnJyAgICAgICAgY29uc3QgY2hhciogZ3B1X2xheWVyc19lbnYgPSBzdGQ6OmdldGVudigiTkVNT1RST05fR1BVX0xBWUVSUyIpOw0KICAgICAgICBpZiAoZ3B1X2xheWVyc19lbnYpIHsNCiAgICAgICAgICAgIGNoYXIqIHBhcnNlZF9lbmQgPSBudWxscHRyOw0KICAgICAgICAgICAgY29uc3QgbG9uZyBncHVfbGF5ZXJzID0gc3RkOjpzdHJ0b2woZ3B1X2xheWVyc19lbnYsICZwYXJzZWRfZW5kLCAxMCk7DQogICAgICAgICAgICBpZiAocGFyc2VkX2VuZCA9PSBncHVfbGF5ZXJzX2VudiB8fCAqcGFyc2VkX2VuZCAhPSAnXFwwJyB8fCBncHVfbGF5ZXJzIDwgMCB8fCBncHVfbGF5ZXJzID4gOTk5KQ0KICAgICAgICAgICAgICAgIHRocm93IHN0ZDo6aW52YWxpZF9hcmd1bWVudCgiTkVNT1RST05fR1BVX0xBWUVSUyBtdXN0IGJlIGFuIGludGVnZXIgZnJvbSAwIHRvIDk5OSIpOw0KICAgICAgICAgICAgbXAubl9ncHVfbGF5ZXJzID0gc3RhdGljX2Nhc3Q8aW50PihncHVfbGF5ZXJzKTsNCiAgICAgICAgfSBlbHNlIHsNCiAgICAgICAgICAgIG1wLm5fZ3B1X2xheWVycyA9IGNmZy5ncHUgPj0gMCA/IDk5OSA6IDA7DQogICAgICAgIH0nJycNCiAgICBpZiAiTkVNT1RST05fR1BVX0xBWUVSUyIgbm90IGluIHNvdXJjZToNCiAgICAgICAgaWYgc291cmNlLmNvdW50KG9sZCkgIT0gMToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUGlubmVkIFNBTE0gc291cmNlIG5vIGxvbmdlciBoYXMgZXhhY3RseSBvbmUgZXhwZWN0ZWQgbl9ncHVfbGF5ZXJzIGFzc2lnbm1lbnQuIikNCiAgICAgICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeCgiLmNwcC50bXAiKQ0KICAgICAgICB0bXAud3JpdGVfdGV4dChzb3VyY2UucmVwbGFjZShvbGQsIG5ldywgMSksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgICAgIHRtcC5yZXBsYWNlKHBhdGgpDQogICAgZWxpZiAibXAubl9ncHVfbGF5ZXJzID0gc3RhdGljX2Nhc3Q8aW50PihncHVfbGF5ZXJzKTsiIG5vdCBpbiBzb3VyY2U6DQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiRXhpc3RpbmcgTkVNT1RST05fR1BVX0xBWUVSUyBwYXRjaCBkb2VzIG5vdCBtYXRjaCB0aGUgZXhwZWN0ZWQgaW1wbGVtZW50YXRpb24uIikNCiAgICBkaWZmID0gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiZGlmZiIsICItLWJpbmFyeSIsICItLSIsICJzcmMvc3BlZWNobG0vc2FsbV9lbmdpbmUuY3BwIl0sDQogICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1SRVBPLCBjaGVjaz1UcnVlLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dA0KICAgIGlmIG5vdCBkaWZmOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIHNvdXJjZSBkaWZmIHdhcyBwcm9kdWNlZCBmb3IgdGhlIEdQVS1sYXllciBvdmVycmlkZS4iKQ0KICAgIFBBVENIX0ZJTEUud3JpdGVfdGV4dChkaWZmLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgIHN0YXRlWyJwYXRjaF9maWxlIl0gPSBzdHIoUEFUQ0hfRklMRSkNCiAgICBzdGF0ZVsic3RhZ2VzIl1bbmFtZV0udXBkYXRlKHsic3RhdHVzIjogImNvbXBsZXRlIiwgImZpbmlzaGVkX2F0X3V0YyI6IHV0YygpLCAiZGlmZl9ieXRlcyI6IGxlbihkaWZmLmVuY29kZSgpKX0pDQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHByaW50KGYiW3ByZXBhcmVfY3BwXSBzYXZlZCBzb3VyY2UgZGlmZjoge1BBVENIX0ZJTEV9IiwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgbWFpbigpIC0+IGludDoNCiAgICBSRVBPUlRfRElSLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBPVVQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIGdsb2JhbCBzdGF0ZQ0KICAgIHN0YXRlID0geyJzdGF0dXMiOiAicnVubmluZyIsICJwaGFzZSI6ICJzdGFydHVwIiwgInN0YXJ0ZWRfYXRfdXRjIjogdXRjKCksICJyZXZpc2lvbiI6IFJFVklTSU9OLA0KICAgICAgICAgICAgICJjaGVja3BvaW50Ijogc3RyKFNPVVJDRSksICJxdWFudGl6YXRpb24iOiAiaXE0X25sIiwgImRlY29kZXIiOiBzdHIoREVDT0RFUiksDQogICAgICAgICAgICAgImxvZyI6IHN0cihMT0cpLCAic3RhZ2VzIjoge30sICJlcnJvciI6IE5vbmV9DQogICAgc2F2ZV9zdGF0dXMoKQ0KICAgIHRyeToNCiAgICAgICAgaWYgUkVQTy5leGlzdHMoKSBhbmQgbm90IChSRVBPIC8gIi5naXQiKS5leGlzdHMoKToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSBub24tZ2l0IHBhdGg6IHtSRVBPfSIpDQogICAgICAgIGlmIG5vdCBSRVBPLmV4aXN0cygpOg0KICAgICAgICAgICAgcnVuX3N0YWdlKCJjbG9uZSIsIFsiZ2l0IiwgImNsb25lIiwgIi0tZGVwdGgiLCAiMSIsICJodHRwczovL2dpdGh1Yi5jb20vTlZJRElBL05lTW8tU3BlZWNoLmNwcC5naXQiLCBzdHIoUkVQTyldKQ0KICAgICAgICBoZWFkID0gc3VicHJvY2Vzcy5ydW4oWyJnaXQiLCAiLUMiLCBzdHIoUkVQTyksICJyZXYtcGFyc2UiLCAiSEVBRCJdLCBjaGVjaz1UcnVlLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dC5zdHJpcCgpDQogICAgICAgIGlmIGhlYWQgIT0gUkVWSVNJT046DQogICAgICAgICAgICBkaXJ0eSA9IHN1YnByb2Nlc3MucnVuKFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZGlmZiIsICItLXF1aWV0Il0sIGNoZWNrPUZhbHNlKS5yZXR1cm5jb2RlDQogICAgICAgICAgICBpZiBkaXJ0eToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJSZWZ1c2luZyB0byBjaGVja291dCB7UkVWSVNJT059OyBleGlzdGluZyB0cmFja2VkIGNoYW5nZXMgd291bGQgYmUgbG9zdC4iKQ0KICAgICAgICAgICAgcnVuX3N0YWdlKCJmZXRjaF9yZXZpc2lvbiIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAiZmV0Y2giLCAiLS1kZXB0aCIsICIxIiwgIm9yaWdpbiIsIFJFVklTSU9OXSkNCiAgICAgICAgICAgIHJ1bl9zdGFnZSgiY2hlY2tvdXRfcmV2aXNpb24iLCBbImdpdCIsICItQyIsIHN0cihSRVBPKSwgImNoZWNrb3V0IiwgIi0tZGV0YWNoIiwgUkVWSVNJT05dKQ0KICAgICAgICBydW5fc3RhZ2UoInN1Ym1vZHVsZSIsIFsiZ2l0IiwgIi1DIiwgc3RyKFJFUE8pLCAic3VibW9kdWxlIiwgInVwZGF0ZSIsICItLWluaXQiLCAiLS1kZXB0aCIsICIxIiwgImxsYW1hLmNwcCJdKQ0KICAgICAgICBydW5fc3RhZ2UoInN5c3RlbV9kZXBlbmRlbmNpZXMiLCBbImFwdC1nZXQiLCAiaW5zdGFsbCIsICIteSIsICJidWlsZC1lc3NlbnRpYWwiLCAiY21ha2UiLCAibmluamEtYnVpbGQiLCAiZ2l0IiwgInBrZy1jb25maWciLCAibGlic2VudGVuY2VwaWVjZS1kZXYiXSkNCiAgICAgICAgaWYgbm90IHNodXRpbC53aGljaCgibnZjYyIpOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJudmNjIGlzIG1pc3Npbmc7IHRoZSBDVURBIHRvb2xraXQgbXVzdCBiZSBhdmFpbGFibGUgaW4gdGhlIENvbGFiIHJ1bnRpbWUuIikNCiAgICAgICAgaWYgbm90IFNPVVJDRS5pc19kaXIoKToNCiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRXhwZWN0ZWQgZXhpc3RpbmcgQkYxNiBzbmFwc2hvdDsgcmVmdXNpbmcgbW9kZWwgZG93bmxvYWQ6IHtTT1VSQ0V9IikNCiAgICAgICAgdG9yY2hfdmVyc2lvbiA9IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCJ0b3JjaCIpDQogICAgICAgIGlmIHRvcmNoX3ZlcnNpb24gIT0gIjIuMTEuMCtjdTEzMCI6DQogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFeHBlY3RlZCBleGlzdGluZyB0b3JjaCAyLjExLjArY3UxMzA7IGZvdW5kIHt0b3JjaF92ZXJzaW9ufS4gTm8gcGlwIGluc3RhbGwgd2FzIGF0dGVtcHRlZC4iKQ0KICAgICAgICB0b3JjaGF1ZGlvX3ZlcnNpb24gPSBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbigidG9yY2hhdWRpbyIpDQogICAgICAgIHJlcXVpcmVtZW50cyA9IChSRVBPIC8gInJlcXVpcmVtZW50cy50eHQiKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpDQogICAgICAgIGZpbHRlcmVkID0gWyJnZ3VmPj0wLjEwLjAiLCAic2VudGVuY2VwaWVjZT49MC4xLjk4LDwwLjMuMCIsICJQeVlBTUw+PTYsPDciXQ0KICAgICAgICByZXFfZmlsZSA9IFJFUE9SVF9ESVIgLyAiY3BwLXJlcXVpcmVtZW50cy50eHQiDQogICAgICAgIGNvbnN0cmFpbnRfZmlsZSA9IFJFUE9SVF9ESVIgLyAidG9yY2gtY29uc3RyYWludHMudHh0Ig0KICAgICAgICByZXFfZmlsZS53cml0ZV90ZXh0KCJcbiIuam9pbihmaWx0ZXJlZCkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgICAgICBjb25zdHJhaW50X2ZpbGUud3JpdGVfdGV4dChmInRvcmNoPT17dG9yY2hfdmVyc2lvbn1cbnRvcmNoYXVkaW89PXt0b3JjaGF1ZGlvX3ZlcnNpb259XG5udW1weT09e2ltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCdudW1weScpfVxudHJhbnNmb3JtZXJzPT17aW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24oJ3RyYW5zZm9ybWVycycpfVxuIiwgZW5jb2Rpbmc9InV0Zi04IikNCiAgICAgICAgcnVuX3N0YWdlKCJjb252ZXJ0ZXJfZGVwZW5kZW5jaWVzIiwgW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLS1kaXNhYmxlLXBpcC12ZXJzaW9uLWNoZWNrIiwNCiAgICAgICAgICAgICAgICAgICItLXVwZ3JhZGUtc3RyYXRlZ3kiLCAib25seS1pZi1uZWVkZWQiLCAiLS1jb25zdHJhaW50Iiwgc3RyKGNvbnN0cmFpbnRfZmlsZSksICItciIsIHN0cihyZXFfZmlsZSldLCBjd2Q9UkVQTykNCiAgICAgICAgaWYgaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24oInRvcmNoIikgIT0gdG9yY2hfdmVyc2lvbjoNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiQ29udmVydGVyIGRlcGVuZGVuY3kgaW5zdGFsbCBjaGFuZ2VkIHRoZSBleGlzdGluZyB0b3JjaCB2ZXJzaW9uLiIpDQogICAgICAgIHBhdGNoX2dwdV9sYXllcnMoKQ0KICAgICAgICBjb25maWd1cmUgPSBbInNjcmlwdHMvY29uZmlndXJlLnNoIiwgImN1ZGEtYXNyIiwgIi1ER0dNTF9DVURBPU9OIiwgIi1EQ01BS0VfQ1VEQV9BUkNISVRFQ1RVUkVTPTc1IiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0dHTUxfUEFUQ0hFRD1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9BU1I9T04iLCAiLURORU1PX1NQRUVDSF9CVUlMRF9TUEVFQ0hMTT1PTiIsDQogICAgICAgICAgICAgICAgICAgICAiLURORU1PX1NQRUVDSF9CVUlMRF9ESUFSPU9GRiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX1RUUz1PRkYiLCAiLURORU1PX1NQRUVDSF9CVUlMRF9OTVQ9T0ZGIiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0NMST1PTiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX01JQ19DQVBUVVJFPU9GRiIsICItRE5FTU9fU1BFRUNIX0JVSUxEX0hUVFA9T0ZGIiwNCiAgICAgICAgICAgICAgICAgICAgICItRE5FTU9fU1BFRUNIX0JVSUxEX0dSUEM9T0ZGIiwgIi1ETkVNT19TUEVFQ0hfQlVJTERfUzJTPU9GRiJdDQogICAgICAgIHJ1bl9zdGFnZSgiY29uZmlndXJlIiwgY29uZmlndXJlLCBjd2Q9UkVQTykNCiAgICAgICAgcnVuX3N0YWdlKCJidWlsZCIsIFsiY21ha2UiLCAiLS1idWlsZCIsICItLXByZXNldCIsICJjdWRhLWFzciIsICItLXBhcmFsbGVsIiwgIjgiXSwgY3dkPVJFUE8pDQogICAgICAgIHJ1bl9zdGFnZSgiY29udmVydF9pcTRfbmwiLCBbc3lzLmV4ZWN1dGFibGUsIHN0cihSRVBPIC8gImNvbnZlcnRfbW9kZWwucHkiKSwgc3RyKFNPVVJDRSksICItLWFyY2hpdGVjdHVyZSIsICJzYWxtIiwNCiAgICAgICAgICAgICAgICAgICItLW91dHR5cGUiLCAiaXE0X25sIiwgIi0tb3V0ZmlsZSIsIHN0cihERUNPREVSKV0sIGN3ZD1SRVBPKQ0KICAgICAgICBlbmNvZGVycyA9IGxpc3QoT1VULmdsb2IoIiotRW5jb2Rlci1GMTYuZ2d1ZiIpKQ0KICAgICAgICBpZiBub3QgREVDT0RFUi5pc19maWxlKCkgb3Igbm90IGVuY29kZXJzOg0KICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDb252ZXJ0ZXIgZXhpdGVkIHN1Y2Nlc3NmdWxseSBidXQgZGVjb2RlciBvciBGMTYgc3BlZWNoIGVuY29kZXIgaXMgbWlzc2luZy4iKQ0KICAgICAgICBzdGF0ZS51cGRhdGUoeyJzdGF0dXMiOiAiY29tcGxldGUiLCAicGhhc2UiOiAiY29tcGxldGUiLCAiZmluaXNoZWRfYXRfdXRjIjogdXRjKCksDQogICAgICAgICAgICAgICAgICAgICAgImVuY29kZXJfZmlsZXMiOiBbc3RyKHBhdGgpIGZvciBwYXRoIGluIGVuY29kZXJzXX0pDQogICAgICAgIHNhdmVfc3RhdHVzKCkNCiAgICAgICAgcHJpbnQoZiJbcHJlcGFyZV9jcHBdIGNvbXBsZXRlOiB7REVDT0RFUn0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4gMA0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICBzdGF0ZS51cGRhdGUoeyJzdGF0dXMiOiAiZmFpbGVkIiwgInBoYXNlIjogc3RhdGUuZ2V0KCJwaGFzZSIsICJ1bmtub3duIiksICJmaW5pc2hlZF9hdF91dGMiOiB1dGMoKSwNCiAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiOiB7InR5cGUiOiB0eXBlKGV4YykuX19uYW1lX18sICJtZXNzYWdlIjogc3RyKGV4Yyl9fSkNCiAgICAgICAgc2F2ZV9zdGF0dXMoKQ0KICAgICAgICBwcmludChmIltwcmVwYXJlX2NwcF0gZmFpbGVkIGluIHtzdGF0ZVsncGhhc2UnXX06IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIsIGZpbGU9c3lzLnN0ZGVyciwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmFpc2UNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQ0K').decode('utf-8-sig')
worker = root / 'prepare_cpp.py'
worker.write_text(source)
log = (root / 'cpp-launch.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(worker)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
(root / 'cpp-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(worker)}))
print('Started', worker.name, 'PID', p.pid, flush=True)


Started prepare_cpp.py PID 14924


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "error": null}


prepare_cpp.log been written to: /content/nemo-speech/build/cuda-asr

[2026-10-30T05:54:06.243997+00:00] build: cmake --build --preset cuda-asr --parallel 8
[0/2] Re-checking globbed directories...
[1/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq3_xxs.cu.o
[2/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq4_nl.cu.o
[3/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq2_xxs.cu.o
[4/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq1_s.cu.o
[5/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq3_s.cu.o
[6/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq2_xs.cu.o
[7/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-inst

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "error": null}


prepare_cpp.log ir/template-instances/mmq-instance-iq3_s.cu.o
[6/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq2_xs.cu.o
[7/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq2_s.cu.o
[8/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/fattn-mma-f16-instance-ncols1_8-ncols2_4.cu.o
[9/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q1_0.cu.o
[10/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-iq4_xs.cu.o
[11/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-q2_0.cu.o
[12/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-instance-mxfp4.cu.o
[13/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmq-inst

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "error": null}


prepare_cpp.log mplate-instances/mmf-instance-ncols_12.cu.o
[28/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_13.cu.o
[29/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_14.cu.o
[30/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_15.cu.o
[31/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_16.cu.o
[32/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_2.cu.o
[33/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_3.cu.o
[34/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instance-ncols_4.cu.o
[35/220] Building CUDA object ggml/src/ggml-cuda/CMakeFiles/ggml-cuda.dir/template-instances/mmf-instan

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "build", "error": null}


prepare_cpp.log 0] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/ngram-mod.cpp.o
[128/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/ngram-map.cpp.o
[129/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/download.cpp.o
[130/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/reasoning-budget.cpp.o
[131/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/subproc.cpp.o
[132/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/trie.cpp.o
[133/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/unicode.cpp.o
[134/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/sampling.cpp.o
[135/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/jinja/lexer.cpp.o
[136/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/json.cpp.o
[137/220] Building CXX object llama.cpp/common/CMakeFiles/llama-common.dir/json-schema-t

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log es/ggml-base.dir/gguf.cpp.o
[19/138] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[20/138] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/arch/x86/quants.c.o
[21/138] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/binary-ops.cpp.o
[22/138] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/unary-ops.cpp.o
[23/138] Building CXX object src/CMakeFiles/llama.dir/Unity/unity_9_cxx.cxx.o
[24/138] Building CXX object src/CMakeFiles/llama.dir/Unity/unity_6_cxx.cxx.o
[25/138] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[26/138] Linking CXX shared library bin/libggml-base.so.0.25.1
[27/138] Creating library symlink bin/libggml-base.so.0 bin/libggml-base.so
[28/138] Building CXX object src/CMakeFiles/llama.dir/Unity/unity_7_cxx.cxx.o
[29/138] Building CXX object src/CMakeFiles/llama.dir/Unity/unity_8_cxx.cxx.o
[30/138] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/llamaf

In [ ]:
from pathlib import Path
import os
import subprocess

paths = [path for path in Path('/content/nemo-speech/build').rglob('nemo-speech') if path.is_file() and os.access(path, os.X_OK)]
assert paths
binary = paths[0]
for command in [[str(binary), 'help', 'transcribe'], [str(binary), 'doctor']]:
    result = subprocess.run(command, capture_output=True, text=True, timeout=60)
    print('command', command, 'returncode', result.returncode, flush=True)
    print((result.stdout + result.stderr)[-4000:], flush=True)


command ['/content/nemo-speech/build/cuda-asr/bin/nemo-speech', 'help', 'transcribe'] returncode 0


Usage: /content/nemo-speech/build/cuda-asr/bin/nemo-speech transcribe INPUT [--model MODEL] [options]

Transcribe a WAV file or directory. Directory
work shares one recognizer; --concurrency feeds multiple utterances to
that recognizer so compatible inference is batched on the GPU.

Options:
  -m, --model MODEL         ASR GGUF path or indexed HF repo (speech-LLM: the
                            decoder GGUF; its encoder file next to it is loaded)
                            (default: nvidia/nemotron-3.5-asr-streaming-0.6b)
  --salm-encoder FILE       Speech-LLM encoder GGUF instead of the one the
                            decoder names
  --asr.salm.* VALUE        SALM scheduling settings (config/env also supported)
  -l, --language CODE       Language code (SALM: language hint)
  --instruction TEXT        Speech-LLM: instruction sent with the audio instead of
                            the default prompt, used as written
  --device, --backend DEVICE
                            auto

command ['/content/nemo-speech/build/cuda-asr/bin/nemo-speech', 'doctor'] returncode 0


NeMo-Speech.cpp 0.3.0
Features: asr backend_cuda integrated_vad model_pull punctuation speechlm
Devices:
  [0] gpu            Tesla T4 (14.6 GiB)
  [1] cpu            Intel(R) Xeon(R) CPU @ 2.00GHz (51.0 GiB)
Model downloads: /usr/bin/curl



In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log  Building CXX object common/CMakeFiles/llama-common.dir/json-schema-to-grammar.cpp.o
[122/138] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[123/138] Linking CXX static library vendor/cpp-httplib/libcpp-httplib.a
[124/138] Building CXX object common/CMakeFiles/llama-common.dir/jinja/string.cpp.o
[125/138] Building CXX object tools/quantize/CMakeFiles/llama-quantize.dir/main.cpp.o
[126/138] Building CXX object common/CMakeFiles/llama-common.dir/jinja/parser.cpp.o
[127/138] Building CXX object common/CMakeFiles/llama-common.dir/preset.cpp.o
[128/138] Building CXX object common/CMakeFiles/llama-common.dir/speculative.cpp.o
[129/138] Building CXX object tools/quantize/CMakeFiles/llama-quantize-impl.dir/quantize.cpp.o
[130/138] Building CXX object common/CMakeFiles/llama-common.dir/jinja/caps.cpp.o
[131/138] Building CXX object common/CMakeFiles/llama-common.dir/jinja/runtime.cpp.o
[132/138] Building CXX object common/CMakeFiles/llama-commo

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log prompt %}
    {%- if enable_thinking %}
        {{- '<|im_start|>assistant\n<think>\n' }}
    {%- else %}
        {{- '<|im_start|>assistant\n<think></think>' }}
    {%- endif %}
{%- endif %}
INFO:gguf.gguf_writer:Writing the following files:
INFO:gguf.gguf_writer:/content/nemotron-gguf/decoder-IQ4_NL.gguf.work/decoder-bf16.gguf: n_tensors = 417, total_size = 65.8G

Writing:   0%|          | 0.00/65.8G [00:00<?, ?byte/s]
Writing:   1%|          | 705M/65.8G [00:05<08:36, 126Mbyte/s]
Writing:   2%|▏         | 1.41G/65.8G [00:11<09:11, 117Mbyte/s]
Writing:   2%|▏         | 1.46G/65.8G [00:12<09:30, 113Mbyte/s]
Writing:   2%|▏         | 1.49G/65.8G [00:12<09:26, 114Mbyte/s]
Writing:   4%|▍         | 2.76G/65.8G [00:25<09:54, 106Mbyte/s]
Writing:   6%|▌         | 4.04G/65.8G [00:37<09:53, 104Mbyte/s]
Writing:   6%|▌         | 4.06G/65.8G [00:37<09:58, 103Mbyte/s]
Writing:   6%|▌         | 4.08G/65.8G [00:38<09:56, 104Mbyte/s]
Writing:   6%|▌         | 4.08G/65.8G [00:50<09:

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log | 13.4G/65.8G [02:08<08:23, 104Mbyte/s]
Writing:  20%|██        | 13.4G/65.8G [02:20<08:23, 104Mbyte/s]
Writing:  22%|██▏       | 14.7G/65.8G [02:20<08:19, 102Mbyte/s]
Writing:  22%|██▏       | 14.7G/65.8G [02:21<08:22, 102Mbyte/s]
Writing:  22%|██▏       | 14.7G/65.8G [02:21<08:19, 102Mbyte/s]
Writing:  22%|██▏       | 14.8G/65.8G [02:21<08:12, 104Mbyte/s]
Writing:  23%|██▎       | 14.8G/65.8G [02:21<08:08, 105Mbyte/s]
Writing:  23%|██▎       | 14.8G/65.8G [02:22<08:02, 106Mbyte/s]
Writing:  23%|██▎       | 14.9G/65.8G [02:22<08:03, 106Mbyte/s]
Writing:  23%|██▎       | 14.9G/65.8G [02:22<07:33, 112Mbyte/s]
Writing:  23%|██▎       | 14.9G/65.8G [02:22<07:23, 115Mbyte/s]
Writing:  25%|██▍       | 16.2G/65.8G [02:35<07:53, 105Mbyte/s]
Writing:  27%|██▋       | 17.5G/65.8G [02:47<07:49, 103Mbyte/s]
Writing:  27%|██▋       | 17.5G/65.8G [02:47<07:53, 102Mbyte/s]
Writing:  27%|██▋       | 17.5G/65.8G [02:48<07:52, 102Mbyte/s]
Writing:  27%|██▋       | 17.6G/65.8G [02:48<07:

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log .8G [04:47<06:01, 100Mbyte/s]
Writing:  45%|████▍     | 29.6G/65.8G [05:00<06:01, 100Mbyte/s]
Writing:  47%|████▋     | 30.8G/65.8G [05:00<05:57, 97.9Mbyte/s]
Writing:  47%|████▋     | 30.9G/65.8G [05:00<06:00, 97.1Mbyte/s]
Writing:  47%|████▋     | 30.9G/65.8G [05:00<05:58, 97.5Mbyte/s]
Writing:  47%|████▋     | 30.9G/65.8G [05:01<05:51, 99.4Mbyte/s]
Writing:  47%|████▋     | 31.0G/65.8G [05:01<05:48, 100Mbyte/s] 
Writing:  49%|████▉     | 32.2G/65.8G [05:14<05:37, 99.5Mbyte/s]
Writing:  51%|█████     | 33.5G/65.8G [05:27<05:24, 99.7Mbyte/s]
Writing:  51%|█████     | 33.5G/65.8G [05:27<05:26, 99.0Mbyte/s]
Writing:  51%|█████     | 33.6G/65.8G [05:27<05:24, 99.5Mbyte/s]
Writing:  51%|█████     | 33.6G/65.8G [05:28<05:18, 101Mbyte/s] 
Writing:  51%|█████     | 33.6G/65.8G [05:28<05:16, 102Mbyte/s]
Writing:  51%|█████     | 33.7G/65.8G [05:28<05:12, 103Mbyte/s]
Writing:  51%|█████     | 33.7G/65.8G [05:28<05:08, 104Mbyte/s]
Writing:  51%|█████     | 33.7G/65.8G [05:40<05:

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log .8G [06:50<03:53, 103Mbyte/s]
Writing:  63%|██████▎   | 41.8G/65.8G [06:50<03:50, 105Mbyte/s]
Writing:  65%|██████▌   | 43.0G/65.8G [07:04<04:10, 91.1Mbyte/s]
Writing:  67%|██████▋   | 44.3G/65.8G [07:17<03:49, 93.8Mbyte/s]
Writing:  67%|██████▋   | 44.3G/65.8G [07:18<03:50, 93.1Mbyte/s]
Writing:  67%|██████▋   | 44.4G/65.8G [07:18<03:49, 93.6Mbyte/s]
Writing:  67%|██████▋   | 44.4G/65.8G [07:18<03:43, 95.7Mbyte/s]
Writing:  68%|██████▊   | 44.4G/65.8G [07:18<03:42, 96.2Mbyte/s]
Writing:  68%|██████▊   | 44.5G/65.8G [07:19<03:39, 97.6Mbyte/s]
Writing:  68%|██████▊   | 44.5G/65.8G [07:19<03:35, 99.1Mbyte/s]
Writing:  68%|██████▊   | 44.5G/65.8G [07:30<03:35, 99.1Mbyte/s]
Writing:  70%|██████▉   | 45.8G/65.8G [07:31<03:14, 103Mbyte/s] 
Writing:  71%|███████▏  | 47.0G/65.8G [07:44<03:04, 102Mbyte/s]
Writing:  71%|███████▏  | 47.1G/65.8G [07:44<03:07, 100Mbyte/s]
Writing:  72%|███████▏  | 47.1G/65.8G [07:44<03:06, 100Mbyte/s]
Writing:  72%|███████▏  | 47.1G/65.8G [07:45<03:

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "convert_iq4_nl", "error": null}


prepare_cpp.log 57.8G/65.8G [09:40<01:18, 103Mbyte/s]
Writing:  90%|████████▉ | 59.1G/65.8G [09:41<01:04, 104Mbyte/s]
Writing:  92%|█████████▏| 60.4G/65.8G [09:54<00:53, 101Mbyte/s]
Writing:  92%|█████████▏| 60.4G/65.8G [09:54<00:54, 100Mbyte/s]
Writing:  92%|█████████▏| 60.4G/65.8G [09:54<00:53, 101Mbyte/s]
Writing:  92%|█████████▏| 60.5G/65.8G [09:55<00:52, 103Mbyte/s]
Writing:  92%|█████████▏| 60.5G/65.8G [09:55<00:51, 103Mbyte/s]
Writing:  94%|█████████▍| 61.8G/65.8G [10:07<00:39, 103Mbyte/s]
Writing:  94%|█████████▍| 61.8G/65.8G [10:20<00:39, 103Mbyte/s]
Writing:  96%|█████████▌| 63.1G/65.8G [10:20<00:27, 101Mbyte/s]
Writing:  96%|█████████▌| 63.1G/65.8G [10:20<00:27, 99.9Mbyte/s]
Writing:  96%|█████████▌| 63.1G/65.8G [10:21<00:27, 100Mbyte/s] 
Writing:  96%|█████████▌| 63.2G/65.8G [10:21<00:26, 102Mbyte/s]
Writing:  96%|█████████▌| 63.2G/65.8G [10:21<00:25, 103Mbyte/s]
Writing:  96%|█████████▌| 63.2G/65.8G [10:21<00:25, 105Mbyte/s]
Writing:  96%|█████████▌| 63.2G/65.8G [10:21<00:

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "failed", "phase": "convert_iq4_nl", "error": {"type": "UnicodeDecodeError", "message": "'utf-8' codec can't decode byte 0xc4 in position 128: invalid continuation byte"}}


prepare_cpp.log l_loader: - kv  32:        nemotron_h_moe.expert_weights_scale f32              = 2.500000
llama_model_loader: - kv  33:        nemotron_h_moe.nextn_predict_layers u32              = 1
llama_model_loader: - kv  34:                  nemo_speech.checkpoint_id str              = 0b5820fad1103208f65895b586bd65d2
llama_model_loader: - kv  35:                   nemo_speech.speech_model str              = c17667641642c4265ff2b7037dd101b7d4567...
llama_model_loader: - kv  36:               general.quantization_version u32              = 2
llama_model_loader: - kv  37:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  38:                         tokenizer.ggml.pre str              = pixtral
llama_model_loader: - kv  39:                      tokenizer.ggml.tokens arr[str,131072]  = ["<unk>", "<s>", "</s>", "[INST]", "[...
llama_model_loader: - kv  40:                  tokenizer.ggml.token_type arr[i32,131072]  = [3, 3, 3, 3, 3, 3, 3, 3, 

In [ ]:
import base64, json, pathlib, subprocess, sys
root = pathlib.Path('/content/nemotron-transcribe')
source = base64.b64decode('aW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN5cwppbXBvcnQgdHJhY2ViYWNrCgpyb290ID0gUGF0aCgnL2NvbnRlbnQvbmVtb3Ryb24tdHJhbnNjcmliZScpCnJlcG8gPSBQYXRoKCcvY29udGVudC9uZW1vLXNwZWVjaCcpCm91dGZpbGUgPSBQYXRoKCcvY29udGVudC9uZW1vdHJvbi1nZ3VmL2RlY29kZXItSVE0X05MLmdndWYnKQpiZjE2ID0gb3V0ZmlsZS53aXRoX25hbWUob3V0ZmlsZS5uYW1lICsgJy53b3JrJykgLyAnZGVjb2Rlci1iZjE2LmdndWYnCmFzc2VydCBiZjE2LmlzX2ZpbGUoKSBhbmQgYmYxNi5zdGF0KCkuc3Rfc2l6ZSA+IDYwXzAwMF8wMDBfMDAwCnN0YXRlX3BhdGggPSByb290IC8gJ2NwcC1zdGF0dXMuanNvbicKc3RhdGUgPSBqc29uLmxvYWRzKHN0YXRlX3BhdGgucmVhZF90ZXh0KCkpCnNodXRpbC5jb3B5ZmlsZShzdGF0ZV9wYXRoLCByb290IC8gJ2xvZy1kZWNvZGluZy1hdHRlbXB0LWNwcC1zdGF0dXMuanNvbicpCnN0YXRlLnVwZGF0ZShzdGF0dXM9J3J1bm5pbmcnLCBwaGFzZT0ncmVzdW1lX3F1YW50aXplJywgZXJyb3I9Tm9uZSkKc3RhdGVfcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUsIGluZGVudD0yKSkKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihyZXBvKSkKZnJvbSBjb252ZXJzaW9uLnNhbG0gaW1wb3J0IHF1YW50aXplX2RlY29kZXIKZnJvbSBjb252ZXJzaW9uLnMyc19jb21wb25lbnRzLnZvaWNlY2hhdF9zb3VyY2UgaW1wb3J0IGVuc3VyZV9xdWFudGl6ZXIKCnRyeToKICAgIHF1YW50aXplciA9IGVuc3VyZV9xdWFudGl6ZXIocmVwbyAvICdsbGFtYS5jcHAnLCBOb25lLCBwcm9maWxlPSdpcTRfbmwnLCByZXBvX3Jvb3Q9cmVwbywgYXV0b19idWlsZD1UcnVlKQogICAgcHJpbnQoJ1Jlc3VtaW5nIG9ubHkgSVE0X05MIHF1YW50aXphdGlvbjsgcmV1c2luZyBjb21wbGV0ZWQgQkYxNiBHR1VGLicsIGZsdXNoPVRydWUpCiAgICBxdWFudGl6ZV9kZWNvZGVyKGJmMTYsIG91dGZpbGUsICdpcTRfbmwnLCBxdWFudGl6ZXIsICdxOF8wJykKICAgIGVuY29kZXJzID0gbGlzdChvdXRmaWxlLnBhcmVudC5nbG9iKCcqLUVuY29kZXItRjE2LmdndWYnKSkKICAgIGFzc2VydCBvdXRmaWxlLmlzX2ZpbGUoKSBhbmQgZW5jb2RlcnMKICAgIHN0YXRlLnVwZGF0ZShzdGF0dXM9J2NvbXBsZXRlJywgcGhhc2U9J2NvbXBsZXRlJywgZGVjb2Rlcl9ieXRlcz1vdXRmaWxlLnN0YXQoKS5zdF9zaXplLCBlbmNvZGVyX2ZpbGVzPVtzdHIocGF0aCkgZm9yIHBhdGggaW4gZW5jb2RlcnNdKQogICAgc3RhdGVfcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUsIGluZGVudD0yKSkKICAgIHByaW50KCdJUTRfTkwgY29udmVyc2lvbiBjb21wbGV0ZS4nLCBvdXRmaWxlLnN0YXQoKS5zdF9zaXplLCBmbHVzaD1UcnVlKQpleGNlcHQgRXhjZXB0aW9uOgogICAgc3RhdGUudXBkYXRlKHN0YXR1cz0nZmFpbGVkJywgcGhhc2U9J3Jlc3VtZV9xdWFudGl6ZScsIGVycm9yPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICBzdGF0ZV9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdGF0ZSwgaW5kZW50PTIpKQogICAgcmFpc2UK').decode('utf-8-sig')
worker = root / 'resume_quantize.py'
worker.write_text(source)
log = (root / 'cpp-launch-resume.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(worker)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
(root / 'cpp-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(worker)}))
print('Started', worker.name, 'PID', p.pid, flush=True)


Started resume_quantize.py PID 23343


In [ ]:
import json
from pathlib import Path
import psutil

current_pid = json.loads(Path('/content/nemotron-transcribe/cpp-process.json').read_text())['pid']
current = psutil.Process(current_pid)
current_tree = {current_pid} | {process.pid for process in current.children(recursive=True)}
workers = []
for process in psutil.process_iter(['pid', 'ppid', 'cmdline']):
    command = process.info['cmdline'] or []
    if any(arg == '/content/nemo-speech/convert_model.py' or arg.endswith('/llama-quantize') for arg in command):
        workers.append({'pid': process.pid, 'ppid': process.ppid(), 'current_worker': process.pid in current_tree, 'command': command})
print(json.dumps({'current_pid': current_pid, 'quantizers': workers, 'bf16_exists': Path('/content/nemotron-gguf/decoder-IQ4_NL.gguf.work/decoder-bf16.gguf').exists()}, indent=2), flush=True)
assert all(worker['current_worker'] for worker in workers), 'Old conversion worker still exists; do not proceed until isolated'


{
  "current_pid": 23343,
  "quantizers": [
    {
      "pid": 23351,
      "ppid": 23343,
      "current_worker": true,
      "command": [
        "/content/nemo-speech/llama.cpp/build-quantize/bin/llama-quantize",
        "--tensor-type",
        "ffn_(up|down|gate)_exps=iq4_nl",
        "--tensor-type",
        "ssm_(in|out)=q8_0",
        "--tensor-type",
        "ffn_(up|down)_shexp=q8_0",
        "--tensor-type",
        "attn_(q|k|v|output)=q8_0",
        "--tensor-type",
        ".=bf16",
        "/content/nemotron-gguf/decoder-IQ4_NL.gguf.work/decoder-bf16.gguf",
        "/content/nemotron-gguf/decoder-IQ4_NL.gguf.quantizing",
        "Q8_0",
        "8"
      ]
    }
  ],
  "bf16_exists": true
}


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


prepare_cpp.log l_loader: - kv  32:        nemotron_h_moe.expert_weights_scale f32              = 2.500000
llama_model_loader: - kv  33:        nemotron_h_moe.nextn_predict_layers u32              = 1
llama_model_loader: - kv  34:                  nemo_speech.checkpoint_id str              = 0b5820fad1103208f65895b586bd65d2
llama_model_loader: - kv  35:                   nemo_speech.speech_model str              = c17667641642c4265ff2b7037dd101b7d4567...
llama_model_loader: - kv  36:               general.quantization_version u32              = 2
llama_model_loader: - kv  37:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  38:                         tokenizer.ggml.pre str              = pixtral
llama_model_loader: - kv  39:                      tokenizer.ggml.tokens arr[str,131072]  = ["<unk>", "<s>", "</s>", "[INST]", "[...
llama_model_loader: - kv  40:                  tokenizer.ggml.token_type arr[i32,131072]  = [3, 3, 3, 3, 3, 3, 3, 3, 

cpp-launch-resume.log  1], type =    f32, size =    0.016 MiB
[  12/ 417] blk.0.ssm_out.weight                 - [  4096,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    21.00 MiB ->    11.16 MiB
[  13/ 417] blk.1.attn_norm.weight               - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[  14/ 417] blk.1.exp_probs_b.bias               - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[  15/ 417] blk.1.ffn_down_exps.weight           - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[  16/ 417] blk.1.ffn_down_shexp.weight          - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[  17/ 417] blk.1.ffn_gate_inp.weight            - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[  18/ 417] blk.1.ffn_up_exps.weight             - [  2688,   1856,    128,      1], type =   bf16, co

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


prepare_cpp.log l_loader: - kv  32:        nemotron_h_moe.expert_weights_scale f32              = 2.500000
llama_model_loader: - kv  33:        nemotron_h_moe.nextn_predict_layers u32              = 1
llama_model_loader: - kv  34:                  nemo_speech.checkpoint_id str              = 0b5820fad1103208f65895b586bd65d2
llama_model_loader: - kv  35:                   nemo_speech.speech_model str              = c17667641642c4265ff2b7037dd101b7d4567...
llama_model_loader: - kv  36:               general.quantization_version u32              = 2
llama_model_loader: - kv  37:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  38:                         tokenizer.ggml.pre str              = pixtral
llama_model_loader: - kv  39:                      tokenizer.ggml.tokens arr[str,131072]  = ["<unk>", "<s>", "</s>", "[INST]", "[...
llama_model_loader: - kv  40:                  tokenizer.ggml.token_type arr[i32,131072]  = [3, 3, 3, 3, 3, 3, 3, 3, 

cpp-launch-resume.log _0 .. size =     1.31 MiB ->     0.70 MiB
[  46/ 417] blk.5.attn_norm.weight               - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[  47/ 417] blk.5.attn_output.weight             - [  4096,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    21.00 MiB ->    11.16 MiB
[  48/ 417] blk.5.attn_q.weight                  - [  2688,   4096,      1,      1], type =   bf16, converting to q8_0 .. size =    21.00 MiB ->    11.16 MiB
[  49/ 417] blk.5.attn_v.weight                  - [  2688,    256,      1,      1], type =   bf16, converting to q8_0 .. size =     1.31 MiB ->     0.70 MiB
[  50/ 417] blk.6.attn_norm.weight               - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[  51/ 417] blk.6.exp_probs_b.bias               - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[  52/ 417] blk.6.ffn_down_exps.weight           - [  1856,   2688,    128,      1], type =   bf16, co

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


prepare_cpp.log l_loader: - kv  32:        nemotron_h_moe.expert_weights_scale f32              = 2.500000
llama_model_loader: - kv  33:        nemotron_h_moe.nextn_predict_layers u32              = 1
llama_model_loader: - kv  34:                  nemo_speech.checkpoint_id str              = 0b5820fad1103208f65895b586bd65d2
llama_model_loader: - kv  35:                   nemo_speech.speech_model str              = c17667641642c4265ff2b7037dd101b7d4567...
llama_model_loader: - kv  36:               general.quantization_version u32              = 2
llama_model_loader: - kv  37:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  38:                         tokenizer.ggml.pre str              = pixtral
llama_model_loader: - kv  39:                      tokenizer.ggml.tokens arr[str,131072]  = ["<unk>", "<s>", "</s>", "[INST]", "[...
llama_model_loader: - kv  40:                  tokenizer.ggml.token_type arr[i32,131072]  = [3, 3, 3, 3, 3, 3, 3, 3, 

cpp-launch-resume.log          - [     1,     64,      1,      1], type =    f32, size =    0.000 MiB
[  62/ 417] blk.7.ssm_dt.bias                    - [    64,      1,      1,      1], type =    f32, size =    0.000 MiB
[  63/ 417] blk.7.ssm_in.weight                  - [  2688,  10304,      1,      1], type =   bf16, converting to q8_0 .. size =    52.83 MiB ->    28.06 MiB
[  64/ 417] blk.7.ssm_norm.weight                - [   512,      8,      1,      1], type =    f32, size =    0.016 MiB
[  65/ 417] blk.7.ssm_out.weight                 - [  4096,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    21.00 MiB ->    11.16 MiB
[  66/ 417] blk.8.attn_norm.weight               - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[  67/ 417] blk.8.exp_probs_b.bias               - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[  68/ 417] blk.8.ffn_down_exps.weight           - [  1856,   2688,    128,      1], type =   bf16, co

In [ ]:
import json
import os
import psutil

print(json.dumps({'cpus': os.cpu_count(), 'affinity': len(os.sched_getaffinity(0)), 'cpu_utilization': psutil.cpu_percent(interval=1, percpu=True), 'available_ram_gib': psutil.virtual_memory().available / 1024**3, 'load': os.getloadavg()}), flush=True)


{"cpus": 8, "affinity": 8, "cpu_utilization": [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0], "available_ram_gib": 46.190677642822266, "load": [7.58740234375, 6.296875, 5.15380859375]}


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['prepare_cpp.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


prepare_cpp.log l_loader: - kv  32:        nemotron_h_moe.expert_weights_scale f32              = 2.500000
llama_model_loader: - kv  33:        nemotron_h_moe.nextn_predict_layers u32              = 1
llama_model_loader: - kv  34:                  nemo_speech.checkpoint_id str              = 0b5820fad1103208f65895b586bd65d2
llama_model_loader: - kv  35:                   nemo_speech.speech_model str              = c17667641642c4265ff2b7037dd101b7d4567...
llama_model_loader: - kv  36:               general.quantization_version u32              = 2
llama_model_loader: - kv  37:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: - kv  38:                         tokenizer.ggml.pre str              = pixtral
llama_model_loader: - kv  39:                      tokenizer.ggml.tokens arr[str,131072]  = ["<unk>", "<s>", "</s>", "[INST]", "[...
llama_model_loader: - kv  40:                  tokenizer.ggml.token_type arr[i32,131072]  = [3, 3, 3, 3, 3, 3, 3, 3, 

cpp-launch-resume.log  1], type =    f32, size =    0.016 MiB
[  81/ 417] blk.9.ssm_out.weight                 - [  4096,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    21.00 MiB ->    11.16 MiB
[  82/ 417] blk.10.attn_norm.weight              - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[  83/ 417] blk.10.exp_probs_b.bias              - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[  84/ 417] blk.10.ffn_down_exps.weight          - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[  85/ 417] blk.10.ffn_down_shexp.weight         - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[  86/ 417] blk.10.ffn_gate_inp.weight           - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[  87/ 417] blk.10.ffn_up_exps.weight            - [  2688,   1856,    128,      1], type =   bf16, co

In [ ]:
import base64, json, pathlib, subprocess, sys
root = pathlib.Path('/content/nemotron-transcribe')
source = base64.b64decode('aW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0aW1lCmltcG9ydCB0cmFjZWJhY2sKaW1wb3J0IHBzdXRpbAoKcm9vdCA9IFBhdGgoJy9jb250ZW50L25lbW90cm9uLXRyYW5zY3JpYmUnKQpyZXBvID0gUGF0aCgnL2NvbnRlbnQvbmVtby1zcGVlY2gnKQpiZjE2ID0gUGF0aCgnL2NvbnRlbnQvbmVtb3Ryb24tZ2d1Zi9kZWNvZGVyLUlRNF9OTC5nZ3VmLndvcmsvZGVjb2Rlci1iZjE2LmdndWYnKQpvdXQgPSBQYXRoKCcvY29udGVudC9uZW1vdHJvbi1nZ3VmL2RlY29kZXItUTRfMC5nZ3VmJykKb2xkX3dvcmtlciA9IGpzb24ubG9hZHMoKHJvb3QgLyAnY3BwLXByb2Nlc3MuanNvbicpLnJlYWRfdGV4dCgpKVsncGlkJ10KcGF1c2VkID0gW10KZm9yIHByb2Nlc3MgaW4gcHN1dGlsLlByb2Nlc3Mob2xkX3dvcmtlcikuY2hpbGRyZW4ocmVjdXJzaXZlPVRydWUpOgogICAgY29tbWFuZCA9IHByb2Nlc3MuY21kbGluZSgpCiAgICBpZiBjb21tYW5kIGFuZCBjb21tYW5kWzBdLmVuZHN3aXRoKCcvbGxhbWEtcXVhbnRpemUnKToKICAgICAgICBvcy5raWxsKHByb2Nlc3MucGlkLCBzaWduYWwuU0lHU1RPUCkKICAgICAgICBwYXVzZWQuYXBwZW5kKHByb2Nlc3MucGlkKQoocm9vdCAvICdpcTQtcGF1c2VkLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyd3b3JrZXJfcGlkJzogb2xkX3dvcmtlciwgJ3F1YW50aXplcl9waWRzJzogcGF1c2VkLCAncmVhc29uJzogJ1RyeSBmYXN0ZXIgZXhwZXJpbWVudGFsIFE0XzAgZXhwZXJ0IHF1YW50aXphdGlvbiBiZWZvcmUgY29udGludWluZyB0aGUgc3RhbmRhcmQgSVE0X05MIHByb2ZpbGUuJ30sIGluZGVudD0yKSkKc3RhdGUgPSB7J3N0YXR1cyc6ICdydW5uaW5nJywgJ3Byb2ZpbGUnOiAnZXhwZXJpbWVudGFsIFE0XzAgZXhwZXJ0cyAvIFE4XzAgZGVuc2UgLyBCRjE2IHJlbWFpbmRlcicsICdzdGFydGVkX2F0JzogdGltZS50aW1lKCksICdzb3VyY2UnOiBzdHIoYmYxNiksICdkZWNvZGVyJzogc3RyKG91dCksICdlcnJvcic6IE5vbmV9CnN0YXR1cyA9IHJvb3QgLyAnZmFzdC1xdWFudGl6ZS1zdGF0dXMuanNvbicKc3RhdHVzLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdGF0ZSwgaW5kZW50PTIpKQp0cnk6CiAgICBxdWFudGl6ZXIgPSByZXBvIC8gJ2xsYW1hLmNwcC9idWlsZC1xdWFudGl6ZS9iaW4vbGxhbWEtcXVhbnRpemUnCiAgICBjb21tYW5kID0gW3N0cihxdWFudGl6ZXIpLCAnLS10ZW5zb3ItdHlwZScsICdmZm5fKHVwfGRvd258Z2F0ZSlfZXhwcz1xNF8wJywgJy0tdGVuc29yLXR5cGUnLCAnc3NtXyhpbnxvdXQpPXE4XzAnLCAnLS10ZW5zb3ItdHlwZScsICdmZm5fKHVwfGRvd24pX3NoZXhwPXE4XzAnLCAnLS10ZW5zb3ItdHlwZScsICdhdHRuXyhxfGt8dnxvdXRwdXQpPXE4XzAnLCAnLS10ZW5zb3ItdHlwZScsICcuPWJmMTYnLCBzdHIoYmYxNiksIHN0cihvdXQpLCAnUThfMCcsICc4J10KICAgIHN0YXRlWydjb21tYW5kJ10gPSBjb21tYW5kCiAgICBzdGF0dXMud3JpdGVfdGV4dChqc29uLmR1bXBzKHN0YXRlLCBpbmRlbnQ9MikpCiAgICBzdWJwcm9jZXNzLnJ1bihjb21tYW5kLCBjaGVjaz1UcnVlKQogICAgc3RhdGUudXBkYXRlKHN0YXR1cz0nY29tcGxldGUnLCBkZWNvZGVyX2J5dGVzPW91dC5zdGF0KCkuc3Rfc2l6ZSwgZmluaXNoZWRfYXQ9dGltZS50aW1lKCkpCiAgICBzdGF0dXMud3JpdGVfdGV4dChqc29uLmR1bXBzKHN0YXRlLCBpbmRlbnQ9MikpCiAgICBwcmludCgnRXhwZXJpbWVudGFsIFE0XzAgZXhwZXJ0IGRlY29kZXIgcmVhZHkuJywgb3V0LnN0YXQoKS5zdF9zaXplLCBmbHVzaD1UcnVlKQpleGNlcHQgRXhjZXB0aW9uOgogICAgc3RhdGUudXBkYXRlKHN0YXR1cz0nZmFpbGVkJywgZXJyb3I9dHJhY2ViYWNrLmZvcm1hdF9leGMoKSkKICAgIHN0YXR1cy53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUsIGluZGVudD0yKSkKICAgIHJhaXNlCg==').decode('utf-8-sig')
worker = root / 'fast_quantize.py'
worker.write_text(source)
log = (root / 'fast-quantize.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(worker)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
(root / 'fast-quantize-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(worker)}))
print('Started', worker.name, 'PID', p.pid, flush=True)


Started fast_quantize.py PID 26596


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'fast-quantize-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['fast-quantize.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


fast-quantize-status.json {"status": "complete", "error": null}


fast-quantize.log weight           - [  2688,   3712,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 413/ 417] blk.52.nextn.eh_proj.weight          - [  5376,   2688,      1,      1], type =   bf16, size =   27.562 MiB
[ 414/ 417] blk.52.nextn.enorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 415/ 417] blk.52.nextn.hnorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 416/ 417] blk.52.nextn.shared_head_norm.weight - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 417/ 417] blk.52.post_attention_norm.weight    - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
llama_model_quantize_impl: model size  = 62794.01 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  = 19402.42 MiB (4.95 BPW)

llama_quantize: quantize time = 203362.13 ms
llama_quantize:    total time = 203362.13 ms
Experimental Q4_0 expert decoder r

cpp-launch-resume.log  .. size =    21.00 MiB ->    11.16 MiB
[ 102/ 417] blk.12.attn_v.weight                 - [  2688,    256,      1,      1], type =   bf16, converting to q8_0 .. size =     1.31 MiB ->     0.70 MiB
[ 103/ 417] blk.13.attn_norm.weight              - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 104/ 417] blk.13.exp_probs_b.bias              - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[ 105/ 417] blk.13.ffn_down_exps.weight          - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[ 106/ 417] blk.13.ffn_down_shexp.weight         - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 107/ 417] blk.13.ffn_gate_inp.weight           - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[ 108/ 417] blk.13.ffn_up_exps.weight            - [  2688,   1856,    128,      1], type =   bf16, co

*File Operation*: `download` on `/content/nemotron-transcribe/fast-quantize-status.json`

In [ ]:
import base64, json, pathlib, subprocess, sys
root = pathlib.Path('/content/nemotron-transcribe')
source = base64.b64decode('aW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgpyb290ID0gUGF0aCgnL2NvbnRlbnQvbmVtb3Ryb24tdHJhbnNjcmliZScpCnJlcG8gPSBQYXRoKCcvY29udGVudC9uZW1vLXNwZWVjaCcpCmJ1aWxkID0gcmVwbyAvICdidWlsZC9jdWRhLWFzcicKYmluYXJpZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiAocmVwbyAvICdidWlsZCcpLnJnbG9iKCduZW1vLXNwZWVjaCcpIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBvcy5hY2Nlc3MocGF0aCwgb3MuWF9PSyldCmFzc2VydCBiaW5hcmllcywgJ25lbW8tc3BlZWNoIGV4ZWN1dGFibGUgbm90IGZvdW5kJwpiaW5hcnkgPSBiaW5hcmllc1swXQptb2RlbCA9IFBhdGgob3MuZW52aXJvbi5nZXQoJ05FTU9UUk9OX0RFQ09ERVJfUEFUSCcsICcvY29udGVudC9uZW1vdHJvbi1nZ3VmL2RlY29kZXItUTRfMC5nZ3VmJykpCmFzc2VydCBtb2RlbC5pc19maWxlKCkKc3RhdGUgPSB7J3N0YXR1cyc6ICdydW5uaW5nJywgJ2JpbmFyeSc6IHN0cihiaW5hcnkpLCAnZGVjb2Rlcic6IHN0cihtb2RlbCksICdkZWNvZGVyX2J5dGVzJzogbW9kZWwuc3RhdCgpLnN0X3NpemUsICdlbmNvZGVyX2ZpbGVzJzoge3N0cihmaWxlKTogZmlsZS5zdGF0KCkuc3Rfc2l6ZSBmb3IgZmlsZSBpbiBtb2RlbC5wYXJlbnQuZ2xvYignKi1FbmNvZGVyLUYxNi5nZ3VmJyl9LCAncXVhbnRpemF0aW9uJzogJ1E0XzAgcm91dGVkIGV4cGVydHMsIFE4XzAgZGVuc2UgbGluZWFycywgQkYxNiBlbWJlZGRpbmdzL2hlYWQsIEYxNiBzcGVlY2ggZW5jb2RlcjsgZXhwZXJpbWVudGFsIHByb2ZpbGUnLCAnZ3B1JzogJ1Rlc2xhIFQ0JywgJ3BhdGNoJzogJ05FTU9UUk9OX0dQVV9MQVlFUlMgb3ZlcnJpZGUgaW4gc2FsbV9lbmdpbmUuY3BwJywgJ3J1bnMnOiBbXX0KcGF0aCA9IHJvb3QgLyAnc21hbGwtZ3B1LXJlc3VsdHMuanNvbicKcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUsIGluZGVudD0yKSkKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIocmVwbyAvICdsbGFtYS5jcHAvZ2d1Zi1weScpKQppbXBvcnQgZ2d1ZgpyZWFkZXIgPSBnZ3VmLkdHVUZSZWFkZXIoc3RyKG1vZGVsKSkKcXVhbnRfdHlwZXMgPSB7fQpleHBlcnRfdHlwZXMgPSB7fQpmb3IgdGVuc29yIGluIHJlYWRlci50ZW5zb3JzOgogICAga2luZCA9IGdndWYuR0dNTFF1YW50aXphdGlvblR5cGUodGVuc29yLnRlbnNvcl90eXBlKS5uYW1lCiAgICBxdWFudF90eXBlc1traW5kXSA9IHF1YW50X3R5cGVzLmdldChraW5kLCAwKSArIDEKICAgIGlmICdleHBzJyBpbiB0ZW5zb3IubmFtZToKICAgICAgICBleHBlcnRfdHlwZXNba2luZF0gPSBleHBlcnRfdHlwZXMuZ2V0KGtpbmQsIDApICsgMQpzdGF0ZVsnZ2d1Zl90ZW5zb3JfdHlwZXMnXSA9IHF1YW50X3R5cGVzCnN0YXRlWydnZ3VmX2V4cGVydF90ZW5zb3JfdHlwZXMnXSA9IGV4cGVydF90eXBlcwpkZWwgcmVhZGVyCgppbnN0cnVjdGlvbiA9ICdUcmFuc2NyaWJlIHRoaXMgSmFwYW5lc2UgYXVkaW8gd2l0aCBwdW5jdHVhdGlvbi4gT21pdCB2ZXJiYWwgZmlsbGVycywgcmVwZWF0ZWQgd29yZHMsIGFuZCBmYWxzZSBzdGFydHMgd2hpbGUgcHJlc2VydmluZyB0aGUgc3BlYWtlclwncyBpbnRlbmRlZCBtZWFuaW5nLiBXcml0ZSBudW1iZXJzIGluIHRoZWlyIHN0YW5kYXJkIHdyaXR0ZW4gZm9ybS4nCmpvYnMgPSBbKCcxNjA3JywgJy9jb250ZW50L2ZsZXVycy1qYS0wMi53YXYnLCA4KSwgKCcxNTE5JywgJy9jb250ZW50L2ZsZXVycy1qYS0wMS53YXYnLCA4KSwgKCcxNjA3JywgJy9jb250ZW50L2ZsZXVycy1qYS0wMi53YXYnLCAwKSwgKCcxNTE5JywgJy9jb250ZW50L2ZsZXVycy1qYS0wMS53YXYnLCAwKV0KZm9yIHNhbXBsZV9pZCwgYXVkaW8sIGxheWVycyBpbiBqb2JzOgogICAgc3RhdGVbJ2N1cnJlbnRfc2FtcGxlJ10gPSBzYW1wbGVfaWQKICAgIHN0YXRlWydjdXJyZW50X2dwdV9sYXllcnMnXSA9IGxheWVycwogICAgcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpKQogICAgY29tbWFuZCA9IFtzdHIoYmluYXJ5KSwgJy0tdmVyYm9zZScsICd0cmFuc2NyaWJlJywgYXVkaW8sICctLW1vZGVsJywgc3RyKG1vZGVsKSwgJy0tZGV2aWNlJywgJ2N1ZGE6MCcsICctbCcsICdqYScsICctLWluc3RydWN0aW9uJywgaW5zdHJ1Y3Rpb24sICctLW5vLXdhcm11cCcsICctLWFzci5iYWNrZW5kLnRocmVhZHMnLCBzdHIobWluKDgsIG9zLmNwdV9jb3VudCgpIG9yIDEpKSwgJy0tYXNyLnNhbG0ubWF4X3NlcXVlbmNlcycsICcxJywgJy0tYXNyLnNhbG0ubl9jdHhfcGVyX3NlcXVlbmNlJywgJzEwMjQnLCAnLS1hc3Iuc2FsbS5tYXhfY29udGV4dCcsICcxMDI0JywgJy0tYXNyLnNhbG0ubl9iYXRjaCcsICcyNTYnLCAnLS1hc3Iuc2FsbS5lbmNvZGVyX21heF9iYXRjaCcsICcxJywgJy0tYXNyLnNhbG0ubXRwX2RyYWZ0JywgJzAnLCAnLS1hc3Iuc2FsbS5tYXhfbmV3X3Rva2VucycsICcxMjgnXQogICAgZW52aXJvbm1lbnQgPSBvcy5lbnZpcm9uLmNvcHkoKQogICAgZW52aXJvbm1lbnRbJ05FTU9UUk9OX0dQVV9MQVlFUlMnXSA9IHN0cihsYXllcnMpCiAgICBlbnZpcm9ubWVudFsnT01QX05VTV9USFJFQURTJ10gPSBzdHIobWluKDgsIG9zLmNwdV9jb3VudCgpIG9yIDEpKQogICAgcmVzZXJ2YXRpb24gPSBOb25lCiAgICBncHVfYnVkZ2V0ID0gTm9uZQogICAgaWYgbGF5ZXJzID09IDA6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJlZV9ieXRlcywgdG90YWxfYnl0ZXMgPSB0b3JjaC5jdWRhLm1lbV9nZXRfaW5mbygpCiAgICAgICAgcmVzZXJ2ZV9ieXRlcyA9IG1heCgwLCBmcmVlX2J5dGVzIC0gNCAqIDEwMjQqKjMpCiAgICAgICAgcmVzZXJ2YXRpb24gPSB0b3JjaC5lbXB0eShyZXNlcnZlX2J5dGVzLCBkdHlwZT10b3JjaC51aW50OCwgZGV2aWNlPSdjdWRhJykKICAgICAgICBncHVfYnVkZ2V0ID0geydtZXRob2QnOiAnUmVzZXJ2ZSBtZW1vcnkgaW4gdGhpcyB3b3JrZXIgd2hpbGUgbmF0aXZlIHRyYW5zY3JpcHRpb24gcnVuczsgdGhpcyBpcyBub3QgYSB0ZXN0IG9uIGEgcGh5c2ljYWwgNEdCIEdQVS4nLCAncmVzZXJ2ZWRfYnl0ZXMnOiByZXNlcnZlX2J5dGVzLCAnZnJlZV9ieXRlc19iZWZvcmVfbmF0aXZlJzogdG9yY2guY3VkYS5tZW1fZ2V0X2luZm8oKVswXSwgJ3RvdGFsX2RldmljZV9ieXRlcyc6IHRvdGFsX2J5dGVzfQogICAgc3Rkb3V0X3BhdGgsIHN0ZGVycl9wYXRoID0gcm9vdCAvIGYnaXE0LXtzYW1wbGVfaWR9LWxheWVyc3tsYXllcnN9LnR4dCcsIHJvb3QgLyBmJ2lxNC17c2FtcGxlX2lkfS1sYXllcnN7bGF5ZXJzfS5sb2cnCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZ3B1X3BlYWtfbWliLCBuYXRpdmVfZ3B1X3BlYWtfbWliLCByc3NfcGVha19raWIgPSAwLCAwLCAwCiAgICB3aXRoIHN0ZG91dF9wYXRoLm9wZW4oJ3cnKSBhcyBzdGRvdXQsIHN0ZGVycl9wYXRoLm9wZW4oJ3cnKSBhcyBzdGRlcnI6CiAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29tbWFuZCwgc3Rkb3V0PXN0ZG91dCwgc3RkZXJyPXN0ZGVyciwgZW52PWVudmlyb25tZW50KQogICAgICAgIHdoaWxlIHByb2Nlc3MucG9sbCgpIGlzIE5vbmU6CiAgICAgICAgICAgIGdwdSA9IHN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1tZW1vcnkudXNlZCcsICctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpCiAgICAgICAgICAgIGlmIGdwdS5yZXR1cm5jb2RlID09IDA6CiAgICAgICAgICAgICAgICBncHVfcGVha19taWIgPSBtYXgoZ3B1X3BlYWtfbWliLCBpbnQoZ3B1LnN0ZG91dC5zdHJpcCgpLnNwbGl0bGluZXMoKVswXSkpCiAgICAgICAgICAgIGdwdV9wcm9jZXNzZXMgPSBzdWJwcm9jZXNzLnJ1bihbJ252aWRpYS1zbWknLCAnLS1xdWVyeS1jb21wdXRlLWFwcHM9cGlkLHVzZWRfZ3B1X21lbW9yeScsICctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpCiAgICAgICAgICAgIGZvciByb3cgaW4gZ3B1X3Byb2Nlc3Nlcy5zdGRvdXQuc3BsaXRsaW5lcygpOgogICAgICAgICAgICAgICAgdmFsdWVzID0gW3ZhbHVlLnN0cmlwKCkgZm9yIHZhbHVlIGluIHJvdy5zcGxpdCgnLCcpXQogICAgICAgICAgICAgICAgaWYgbGVuKHZhbHVlcykgPT0gMiBhbmQgdmFsdWVzWzBdID09IHN0cihwcm9jZXNzLnBpZCkgYW5kIHZhbHVlc1sxXS5pc2RpZ2l0KCk6CiAgICAgICAgICAgICAgICAgICAgbmF0aXZlX2dwdV9wZWFrX21pYiA9IG1heChuYXRpdmVfZ3B1X3BlYWtfbWliLCBpbnQodmFsdWVzWzFdKSkKICAgICAgICAgICAgc3RhdHVzID0gUGF0aChmJy9wcm9jL3twcm9jZXNzLnBpZH0vc3RhdHVzJykKICAgICAgICAgICAgaWYgc3RhdHVzLmV4aXN0cygpOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHByb2Nlc3Nfc3RhdHVzID0gc3RhdHVzLnJlYWRfdGV4dCgpCiAgICAgICAgICAgICAgICBleGNlcHQgRmlsZU5vdEZvdW5kRXJyb3I6CiAgICAgICAgICAgICAgICAgICAgcHJvY2Vzc19zdGF0dXMgPSAnJwogICAgICAgICAgICAgICAgZm9yIGxpbmUgaW4gcHJvY2Vzc19zdGF0dXMuc3BsaXRsaW5lcygpOgogICAgICAgICAgICAgICAgICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgoJ1ZtUlNTOicsICdWbUhXTTonKSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJzc19wZWFrX2tpYiA9IG1heChyc3NfcGVha19raWIsIGludChsaW5lLnNwbGl0KClbMV0pKQogICAgICAgICAgICBpZiB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCA+IDEyMDA6CiAgICAgICAgICAgICAgICBwcm9jZXNzLnRlcm1pbmF0ZSgpCiAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQodGltZW91dD0zMCkKICAgICAgICAgICAgICAgIHN0YXRlWydzdGF0dXMnXSA9ICdmYWlsZWQnCiAgICAgICAgICAgICAgICBzdGF0ZVsnZXJyb3InXSA9ICdTaW5nbGUtc2FtcGxlIGluZmVyZW5jZSBleGNlZWRlZCAxMjAwIHNlY29uZHMnCiAgICAgICAgICAgICAgICBwYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdGF0ZSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikpCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1NpbmdsZS1zYW1wbGUgaW5mZXJlbmNlIGV4Y2VlZGVkIDEyMDAgc2Vjb25kcycpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoMC41KQogICAgcnVuID0geydzYW1wbGVfaWQnOiBzYW1wbGVfaWQsICdncHVfbGF5ZXJzJzogbGF5ZXJzLCAnZ3B1X2J1ZGdldCc6IGdwdV9idWRnZXQsICdjb21tYW5kJzogY29tbWFuZCwgJ3JldHVybmNvZGUnOiBwcm9jZXNzLnJldHVybmNvZGUsICd0cmFuc2NyaXB0Jzogc3Rkb3V0X3BhdGgucmVhZF90ZXh0KCkuc3RyaXAoKSwgJ3dhbGxfc2Vjb25kc19pbmNsdWRpbmdfbW9kZWxfbG9hZCc6IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkLCAnZ3B1X3BlYWtfZGV2aWNlX3VzZWRfbWliX3NhbXBsZWRfNTAwbXMnOiBncHVfcGVha19taWIsICduYXRpdmVfcHJvY2Vzc19ncHVfcGVha19taWJfc2FtcGxlZF81MDBtcyc6IG5hdGl2ZV9ncHVfcGVha19taWIsICdwcm9jZXNzX3Jzc19wZWFrX2tpYl9zYW1wbGVkXzUwMG1zJzogcnNzX3BlYWtfa2liLCAnYXVkaW9fc2hhMjU2JzogaGFzaGxpYi5zaGEyNTYoUGF0aChhdWRpbykucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwgJ3N0ZGVycl9maWxlJzogc3RyKHN0ZGVycl9wYXRoKX0KICAgIGlmIHJlc2VydmF0aW9uIGlzIG5vdCBOb25lOgogICAgICAgIGRlbCByZXNlcnZhdGlvbgogICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgc3RhdGVbJ3J1bnMnXS5hcHBlbmQocnVuKQogICAgc3RhdGVbJ3N0YXR1cyddID0gJ3J1bm5pbmcnIGlmIHByb2Nlc3MucmV0dXJuY29kZSA9PSAwIGFuZCBydW5bJ3RyYW5zY3JpcHQnXSBlbHNlICdmYWlsZWQnCiAgICBwYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdGF0ZSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikpCiAgICBwcmludChqc29uLmR1bXBzKHJ1biwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZmx1c2g9VHJ1ZSkKICAgIGlmIHN0YXRlWydzdGF0dXMnXSA9PSAnZmFpbGVkJzoKICAgICAgICBwcmludChzdGRlcnJfcGF0aC5yZWFkX3RleHQoZXJyb3JzPSdyZXBsYWNlJylbLTUwMDA6XSwgZmx1c2g9VHJ1ZSkKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSB0cmFuc2NyaXB0aW9uIGZhaWxlZDsgcHJlc2VydmVkIGxvZ3MnKQpzdGF0ZVsnc3RhdHVzJ10gPSAnY29tcGxldGUnCnBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHN0YXRlLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSkKcHJpbnQoJ1R3byBKYXBhbmVzZSByZWNvcmRpbmdzIGNvbXBsZXRlZCBvbiBUNDsgYWxzbyBjaGVja2VkIGEgQ1BVLWRlY29kZXIgY29uZmlndXJhdGlvbiB3aXRoIHRoZSBzcGVlY2ggZW5jb2RlciBvbiBHUFUuJywgZmx1c2g9VHJ1ZSkK').decode('utf-8-sig')
worker = root / 'run_small_gpu.py'
worker.write_text(source)
log = (root / 'small-gpu-worker.log').open('w')
p = subprocess.Popen([sys.executable, '-u', str(worker)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
(root / 'small-gpu-process.json').write_text(json.dumps({'pid': p.pid, 'worker': str(worker)}))
print('Started', worker.name, 'PID', p.pid, flush=True)


Started run_small_gpu.py PID 30920


In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'fast-quantize-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['fast-quantize.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


fast-quantize-status.json {"status": "complete", "error": null}


small-gpu-results.json {"status": "running", "current_sample": "1607", "current_gpu_layers": 8, "runs": []}


fast-quantize.log weight           - [  2688,   3712,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 413/ 417] blk.52.nextn.eh_proj.weight          - [  5376,   2688,      1,      1], type =   bf16, size =   27.562 MiB
[ 414/ 417] blk.52.nextn.enorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 415/ 417] blk.52.nextn.hnorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 416/ 417] blk.52.nextn.shared_head_norm.weight - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 417/ 417] blk.52.post_attention_norm.weight    - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
llama_model_quantize_impl: model size  = 62794.01 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  = 19402.42 MiB (4.95 BPW)

llama_quantize: quantize time = 203362.13 ms
llama_quantize:    total time = 203362.13 ms
Experimental Q4_0 expert decoder r

cpp-launch-resume.log  .. size =    21.00 MiB ->    11.16 MiB
[ 102/ 417] blk.12.attn_v.weight                 - [  2688,    256,      1,      1], type =   bf16, converting to q8_0 .. size =     1.31 MiB ->     0.70 MiB
[ 103/ 417] blk.13.attn_norm.weight              - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 104/ 417] blk.13.exp_probs_b.bias              - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[ 105/ 417] blk.13.ffn_down_exps.weight          - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[ 106/ 417] blk.13.ffn_down_shexp.weight         - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 107/ 417] blk.13.ffn_gate_inp.weight           - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[ 108/ 417] blk.13.ffn_up_exps.weight            - [  2688,   1856,    128,      1], type =   bf16, co

small-gpu-worker.log 


iq4-1607-layers8.log ed tensor blk.52.attn_norm.weight (size = 10752 bytes) -- ignoring
model has unused tensor blk.52.attn_q.weight (size = 11698176 bytes) -- ignoring
model has unused tensor blk.52.attn_k.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_v.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_output.weight (size = 11698176 bytes) -- ignoring
model has unused tensor blk.52.post_attention_norm.weight (size = 10752 bytes) -- ignoring
model has unused tensor blk.52.ffn_gate_inp.weight (size = 688128 bytes) -- ignoring
model has unused tensor blk.52.exp_probs_b.bias (size = 512 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_shexp.weight (size = 10601472 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_shexp.weight (size = 10601472

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'fast-quantize-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['fast-quantize.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


fast-quantize-status.json {"status": "complete", "error": null}


small-gpu-results.json {"status": "running", "current_sample": "1519", "current_gpu_layers": 8, "runs": [{"sample_id": "1607", "returncode": 0, "transcript": "興味をそそられる村を30分ほど散策するのは価値があります。", "gpu_peak_device_used_mib_sampled_500ms": 5223}]}


fast-quantize.log weight           - [  2688,   3712,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 413/ 417] blk.52.nextn.eh_proj.weight          - [  5376,   2688,      1,      1], type =   bf16, size =   27.562 MiB
[ 414/ 417] blk.52.nextn.enorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 415/ 417] blk.52.nextn.hnorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 416/ 417] blk.52.nextn.shared_head_norm.weight - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 417/ 417] blk.52.post_attention_norm.weight    - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
llama_model_quantize_impl: model size  = 62794.01 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  = 19402.42 MiB (4.95 BPW)

llama_quantize: quantize time = 203362.13 ms
llama_quantize:    total time = 203362.13 ms
Experimental Q4_0 expert decoder r

cpp-launch-resume.log  .. size =    21.00 MiB ->    11.16 MiB
[ 102/ 417] blk.12.attn_v.weight                 - [  2688,    256,      1,      1], type =   bf16, converting to q8_0 .. size =     1.31 MiB ->     0.70 MiB
[ 103/ 417] blk.13.attn_norm.weight              - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 104/ 417] blk.13.exp_probs_b.bias              - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[ 105/ 417] blk.13.ffn_down_exps.weight          - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[ 106/ 417] blk.13.ffn_down_shexp.weight         - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 107/ 417] blk.13.ffn_gate_inp.weight           - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[ 108/ 417] blk.13.ffn_up_exps.weight            - [  2688,   1856,    128,      1], type =   bf16, co

small-gpu-worker.log  "/content/nemotron-gguf/decoder-Q4_0.gguf", "--device", "cuda:0", "-l", "ja", "--instruction", "Transcribe this Japanese audio with punctuation. Omit verbal fillers, repeated words, and false starts while preserving the speaker's intended meaning. Write numbers in their standard written form.", "--no-warmup", "--asr.backend.threads", "8", "--asr.salm.max_sequences", "1", "--asr.salm.n_ctx_per_sequence", "1024", "--asr.salm.max_context", "1024", "--asr.salm.n_batch", "256", "--asr.salm.encoder_max_batch", "1", "--asr.salm.mtp_draft", "0", "--asr.salm.max_new_tokens", "128"], "returncode": 0, "transcript": "興味をそそられる村を30分ほど散策するのは価値があります。", "wall_seconds_including_model_load": 83.1595320890001, "gpu_peak_device_used_mib_sampled_500ms": 5223, "native_process_gpu_peak_mib_sampled_500ms": 5088, "process_rss_peak_kib_sampled_500ms": 16721840, "audio_sha256": "88391f365cdecc51a56ef5f9322c441c0107ab8d0c6f7a3a09f2334ffe8c56ff", "stderr_file": "/content/nemotron-transcribe/iq

iq4-1607-layers8.log sed tensor blk.52.attn_k.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_v.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_output.weight (size = 11698176 bytes) -- ignoring
model has unused tensor blk.52.post_attention_norm.weight (size = 10752 bytes) -- ignoring
model has unused tensor blk.52.ffn_gate_inp.weight (size = 688128 bytes) -- ignoring
model has unused tensor blk.52.exp_probs_b.bias (size = 512 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_shexp.weight (size = 10601472 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_shexp.weight (size = 10601472 bytes) -- ignoring
........................................................
[salm] path=upstream llama.cpp mtp_draft=0 slots=1 n_batch=256
[nemo-speech] transcr

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
path = root / 'iq4-1607-layers8.log'
lines = [line for line in path.read_text(errors='replace').splitlines() if 'offload' in line.lower() or ('buffer' in line.lower() and ('CUDA' in line or 'CPU' in line))]
(root / 'native-placement.json').write_text(json.dumps({'source_log': path.name, 'matching_lines': lines}, ensure_ascii=False, indent=2))
print('\n'.join(lines), flush=True)


In [ ]:
import os
os.environ['NEMOTRON_STOP_PREFIX'] = 'iq4-incomplete-'
import json
from pathlib import Path
import shutil
import psutil
import os

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'prepare_cpp.log', 'cpp-launch.log']:
    path = root / name
    if path.exists():
        shutil.copyfile(path, root / (os.environ.get('NEMOTRON_STOP_PREFIX', 'dependency-attempt-1-') + name))
worker_info = json.loads((root / 'cpp-process.json').read_text())
pid = worker_info['pid']
try:
    process = psutil.Process(pid)
    assert worker_info['worker'].startswith('/content/nemotron-transcribe/')
    assert worker_info['worker'] in process.cmdline()
    targets = process.children(recursive=True) + [process]
    for target in reversed(targets):
        try:
            target.terminate()
        except psutil.NoSuchProcess:
            pass
    _, alive = psutil.wait_procs(targets, timeout=5)
    for target in alive:
        target.kill()
except psutil.NoSuchProcess:
    pass
print('Stopped only this experiment C++ preparation worker and its children.', flush=True)


Stopped only this experiment C++ preparation worker and its children.


*File Operation*: `upload` on `/content/nemotron-transcribe/convert_q4_profile.py`

In [ ]:
from pathlib import Path
import subprocess
import sys

source = '/content/hf-cache/models--nvidia--NVIDIA-Nemotron-3.5-Transcribe-30B-A4B/snapshots/c17667641642c4265ff2b7037dd101b7d4567634'
result = subprocess.run([sys.executable, '/content/nemotron-transcribe/convert_q4_profile.py', source, '/content/q4-dryrun-check/decoder-Q4_0.gguf', '--dry-run'], capture_output=True, text=True, errors='replace')
Path('/content/nemotron-transcribe/q4-adapter-dryrun.log').write_text(result.stdout + result.stderr)
print('Q4 adapter dry-run returncode', result.returncode, flush=True)
print((result.stdout + result.stderr)[-1500:], flush=True)


Q4 adapter dry-run returncode 0


ios = ns.audios + 1 %}
                {%- endif %}
            {%- endfor %}
            {%- if ns.text == "<|audio|>" %}
                {%- set ns.text = "" %}
                {%- set ns.audios = ns.audios + 1 %}
            {%- endif %}
            {%- set locators = (["<|audio|>"] * ns.audios) | join(" ") %}
            {%- set content = (ns.text ~ " " ~ locators) if ns.text else locators %}
        {%- endif %}
        {{- content }}
        {{- '<|im_end|>\n' }}
    {%- elif message.role == "tool" %}
        {%- if loop.previtem and loop.previtem.role != "tool" %}
            {{- '<|im_start|>user\n' }}
        {%- endif %}
        {{- '<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>\n' }}
        {%- if not loop.last and loop.nextitem.role != "tool" %}
            {{- '<|im_end|>\n' }}
        {%- elif loop.last %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- else %}
        {{- '<|im_start|>' + message.role + '\n' + messag

In [ ]:
import json
from pathlib import Path

root = Path('/content/nemotron-transcribe')
for name in ['cpp-status.json', 'fast-quantize-status.json', 'small-gpu-results.json']:
    path = root / name
    if path.exists():
        state = json.loads(path.read_text())
        summary = {key: state[key] for key in ['status', 'phase', 'error', 'current_sample', 'current_gpu_layers'] if key in state}
        if 'runs' in state:
            summary['runs'] = [{key: run.get(key) for key in ['sample_id', 'returncode', 'transcript', 'gpu_peak_device_used_mib_sampled_500ms']} for run in state['runs']]
        print(name, json.dumps(summary, ensure_ascii=False), flush=True)
for name in ['fast-quantize.log', 'cpp-launch-resume.log', 'small-gpu-worker.log', 'iq4-1607-layers8.log', 'iq4-1607-layers0.log']:
    path = root / name
    if path.exists():
        print(name, path.read_text(errors='replace')[-1000:], flush=True)


cpp-status.json {"status": "running", "phase": "resume_quantize", "error": null}


fast-quantize-status.json {"status": "complete", "error": null}


small-gpu-results.json {"status": "complete", "current_sample": "1519", "current_gpu_layers": 0, "runs": [{"sample_id": "1607", "returncode": 0, "transcript": "興味をそそられる村を30分ほど散策するのは価値があります。", "gpu_peak_device_used_mib_sampled_500ms": 5223}, {"sample_id": "1519", "returncode": 0, "transcript": "多くの場合、海外のギャップイヤーコースに入学することで、企画後に実際に大学に進学しやすくなります。", "gpu_peak_device_used_mib_sampled_500ms": 5251}, {"sample_id": "1607", "returncode": 0, "transcript": "興味をそそられる村を30分ほど散策するのは価値があります。", "gpu_peak_device_used_mib_sampled_500ms": 13023}, {"sample_id": "1519", "returncode": 0, "transcript": "多くの場合、海外のギャップイヤーコースに入学することで、企画後に実際に大学に進学しやすくなります。", "gpu_peak_device_used_mib_sampled_500ms": 13027}]}


fast-quantize.log weight           - [  2688,   3712,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 413/ 417] blk.52.nextn.eh_proj.weight          - [  5376,   2688,      1,      1], type =   bf16, size =   27.562 MiB
[ 414/ 417] blk.52.nextn.enorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 415/ 417] blk.52.nextn.hnorm.weight            - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 416/ 417] blk.52.nextn.shared_head_norm.weight - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 417/ 417] blk.52.post_attention_norm.weight    - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
llama_model_quantize_impl: model size  = 62794.01 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  = 19402.42 MiB (4.95 BPW)

llama_quantize: quantize time = 203362.13 ms
llama_quantize:    total time = 203362.13 ms
Experimental Q4_0 expert decoder r

cpp-launch-resume.log  .. size =    21.00 MiB ->    11.16 MiB
[ 102/ 417] blk.12.attn_v.weight                 - [  2688,    256,      1,      1], type =   bf16, converting to q8_0 .. size =     1.31 MiB ->     0.70 MiB
[ 103/ 417] blk.13.attn_norm.weight              - [  2688,      1,      1,      1], type =    f32, size =    0.010 MiB
[ 104/ 417] blk.13.exp_probs_b.bias              - [   128,      1,      1,      1], type =    f32, size =    0.000 MiB
[ 105/ 417] blk.13.ffn_down_exps.weight          - [  1856,   2688,    128,      1], type =   bf16, converting to iq4_nl .. size =  1218.00 MiB ->   342.56 MiB
[ 106/ 417] blk.13.ffn_down_shexp.weight         - [  3712,   2688,      1,      1], type =   bf16, converting to q8_0 .. size =    19.03 MiB ->    10.11 MiB
[ 107/ 417] blk.13.ffn_gate_inp.weight           - [  2688,    128,      1,      1], type =    f32, size =    1.312 MiB
[ 108/ 417] blk.13.ffn_up_exps.weight            - [  2688,   1856,    128,      1], type =   bf16, co

small-gpu-worker.log ctuation. Omit verbal fillers, repeated words, and false starts while preserving the speaker's intended meaning. Write numbers in their standard written form.", "--no-warmup", "--asr.backend.threads", "8", "--asr.salm.max_sequences", "1", "--asr.salm.n_ctx_per_sequence", "1024", "--asr.salm.max_context", "1024", "--asr.salm.n_batch", "256", "--asr.salm.encoder_max_batch", "1", "--asr.salm.mtp_draft", "0", "--asr.salm.max_new_tokens", "128"], "returncode": 0, "transcript": "多くの場合、海外のギャップイヤーコースに入学することで、企画後に実際に大学に進学しやすくなります。", "wall_seconds_including_model_load": 84.44662291299937, "gpu_peak_device_used_mib_sampled_500ms": 13027, "native_process_gpu_peak_mib_sampled_500ms": 2174, "process_rss_peak_kib_sampled_500ms": 19685352, "audio_sha256": "9627b8f5d46806e1a433b443d692fdfb24f26a35c1a47eba26291475fc643d7b", "stderr_file": "/content/nemotron-transcribe/iq4-1519-layers0.log"}
Two Japanese recordings completed on T4; also checked a CPU-decoder configuration with the sp

iq4-1607-layers8.log sed tensor blk.52.attn_k.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_v.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_output.weight (size = 11698176 bytes) -- ignoring
model has unused tensor blk.52.post_attention_norm.weight (size = 10752 bytes) -- ignoring
model has unused tensor blk.52.ffn_gate_inp.weight (size = 688128 bytes) -- ignoring
model has unused tensor blk.52.exp_probs_b.bias (size = 512 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_shexp.weight (size = 10601472 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_shexp.weight (size = 10601472 bytes) -- ignoring
........................................................
[salm] path=upstream llama.cpp mtp_draft=0 slots=1 n_batch=256
[nemo-speech] transcr

iq4-1607-layers0.log d tensor blk.52.attn_k.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_v.weight (size = 731136 bytes) -- ignoring
model has unused tensor blk.52.attn_output.weight (size = 11698176 bytes) -- ignoring
model has unused tensor blk.52.post_attention_norm.weight (size = 10752 bytes) -- ignoring
model has unused tensor blk.52.ffn_gate_inp.weight (size = 688128 bytes) -- ignoring
model has unused tensor blk.52.exp_probs_b.bias (size = 512 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_exps.weight (size = 359202816 bytes) -- ignoring
model has unused tensor blk.52.ffn_down_shexp.weight (size = 10601472 bytes) -- ignoring
model has unused tensor blk.52.ffn_up_shexp.weight (size = 10601472 bytes) -- ignoring
..........................................................
[salm] path=upstream llama.cpp mtp_draft=0 slots=1 n_batch=256
[nemo-speech] transcr

In [ ]:
import hashlib
import json
from pathlib import Path
import shutil
import zipfile

root = Path('/content/nemotron-transcribe')
state = json.loads((root / 'small-gpu-results.json').read_text())
assert state['status'] == 'complete' and len(state['runs']) == 4
assert all(run['returncode'] == 0 and run['transcript'].strip() for run in state['runs'])
assert state['gguf_expert_tensor_types'].get('Q4_0', 0) > 0
assert set(state['gguf_expert_tensor_types']) == {'Q4_0'}
expected = {'1607': '88391f365cdecc51a56ef5f9322c441c0107ab8d0c6f7a3a09f2334ffe8c56ff', '1519': '9627b8f5d46806e1a433b443d692fdfb24f26a35c1a47eba26291475fc643d7b'}
for run in state['runs']:
    assert run['audio_sha256'] == expected[run['sample_id']]
    assert run['native_process_gpu_peak_mib_sampled_500ms'] > 0
    if run['gpu_layers'] == 0:
        assert run['gpu_budget']['reserved_bytes'] > 0
        assert run['gpu_budget']['free_bytes_before_native'] <= 4 * 1024**3 + 1024**2
audio_dir = root / 'audio'
audio_dir.mkdir(exist_ok=True)
for index in [1, 2]:
    shutil.copyfile(f'/content/fleurs-ja-{index:02d}.wav', audio_dir / f'fleurs-ja-{index:02d}.wav')
verification = {'status': 'passed', 'scope': 'Four native transcription results, original input hashes, actual Q4_0 expert tensor types, GPU use and available-memory budget. Not a test on a physical 4GB GPU or a general accuracy/speed benchmark.', 'run_count': 4, 'sample_ids': sorted(expected), 'expert_tensor_types': state['gguf_expert_tensor_types'], 'native_gpu_peak_mib': max(run['native_process_gpu_peak_mib_sampled_500ms'] for run in state['runs']), 'budget4_native_gpu_peak_mib': max(run['native_process_gpu_peak_mib_sampled_500ms'] for run in state['runs'] if run['gpu_layers'] == 0), 'budget4_rss_peak_kib': max(run['process_rss_peak_kib_sampled_500ms'] for run in state['runs'] if run['gpu_layers'] == 0)}
(root / 'verification.json').write_text(json.dumps(verification, ensure_ascii=False, indent=2))
cpp = json.loads((root / 'cpp-status.json').read_text())
cpp.update(status='not_completed', phase='superseded_by_experimental_q4_0', error=None, note='Standard IQ4_NL conversion was stopped after the experimental Q4_0 route completed. It is not reported as a completed IQ4_NL conversion.')
(root / 'cpp-status.json').write_text(json.dumps(cpp, indent=2))
archive = Path('/content/nemotron-small-gpu-artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in root.rglob('*'):
        if path.is_file():
            bundle.write(path, path.relative_to(root))
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print(json.dumps({'verification': verification, 'archive_bytes': archive.stat().st_size, 'archive_sha256': hashlib.sha256(archive.read_bytes()).hexdigest()}, ensure_ascii=False, indent=2), flush=True)


{
  "verification": {
    "status": "passed",
    "scope": "Four native transcription results, original input hashes, actual Q4_0 expert tensor types, GPU use and available-memory budget. Not a test on a physical 4GB GPU or a general accuracy/speed benchmark.",
    "run_count": 4,
    "sample_ids": [
      "1519",
      "1607"
    ],
    "expert_tensor_types": {
      "Q4_0": 48
    },
    "native_gpu_peak_mib": 5116,
    "budget4_native_gpu_peak_mib": 2174,
    "budget4_rss_peak_kib": 19685352
  },
  "archive_bytes": 1183642,
  "archive_sha256": "c0bcfc356ab26e2dc05395176957ed758749091e41a54188aecd2959f0c6bd7f"
}


*File Operation*: `download` on `/content/nemotron-small-gpu-artifacts.zip`